# Computational Approaches to Quantum Many-Body Systems: From Exact Diagonalization to Neural Quantum States

##### This project investigates computational approaches to quantum many-body systems, using the one-dimensional spin-½ antiferromagnetic Heisenberg XXX chain as a controlled testbed. Exact diagonalization provides a baseline, while variational Monte Carlo (VMC), including Markov chain Monte Carlo (MCMC) sampling, and matrix-product-state (MPS)/density matrix renormalization group (DMRG) methods provide complementary approaches to ground-state calculations. The project places particular emphasis on the theoretical foundations and computational implementation of neural quantum states, exploring their variational optimization and comparing neural-network and tensor-network representations of many-body wavefunctions.

## 1. Physical Problem

We consider a one-dimensional isotropic spin-$\frac{1}{2}$ Heisenberg chain. The Hamiltonian is

$$
H = J\sum_{i=1}^{N} \mathbf{S}_i\cdot\mathbf{S}_{i+1},
$$

where

$$
\mathbf{S}_i\cdot\mathbf{S}_{i+1}
=
S_i^xS_{i+1}^x+
S_i^yS_{i+1}^y+
S_i^zS_{i+1}^z.
$$

For the main part of this project we take $J=1$ and $\hbar = 1$ and consider the antiferromagnetic case.

The goal is to study how the ground state of this interacting quantum many-body system can be represented and approximated using different computational approaches, with particular emphasis on neural quantum states and variational Monte Carlo.

We will use exact diagonalization (ED) and matrix-product-state/density-matrix-renormalization-group (MPS/DMRG) calculations as benchmarks.

The central questions of the project are:

1. How can a many-body wavefunction be represented by a neural network?
2. How can its variational parameters be optimized without explicitly constructing the exponentially large Hilbert-space vector?
3. How accurately can a neural quantum state approximate the ground state?
4. How does its performance compare with MPS/DMRG?
5. What can these different representations teach us about the structure of quantum many-body states?


## 2. Exact Diagonalization

### 2.1 Derivation of the Hamiltonian's action on the computational basis 
A prominent feature of the system we are considering is that, since the Hamiltonian contains only nearest-neighbor interactions, each basis state is coupled by $H$ only to itself and to basis states obtained by exchanging neighboring opposite spins. Since there are only $N-1$ nearest-neighbor pairs, each column of the Hamiltonian contains at most $N$ nonzero entries. Consequently, when represented in matrix form, the Hamiltonian is sparse, which greatly simplifies diagonalization and reduces the computational effort significantly. Let us begin by constructing the Hamiltonian. We will attempt to find the matrix form of the Hamiltonian by finding its action on the basis states of the Hilbert space, i.e., our construction will build the Hamiltonian column by column.

Suppose we have a spin chain composed of $N$ spin-$\frac{1}{2}$ particles (electrons, for instance). The spin states are denoted by $\left| \uparrow \right\rangle$ and $\left| \downarrow \right\rangle$ for up spins and down spins, respectively. To represent the states computationally, we will use 1 for $\left|\uparrow\right\rangle$ and 0 for $\left|\downarrow\right\rangle$. So, for instance,

$$
\left| \uparrow \uparrow \uparrow \downarrow \downarrow \right\rangle
\equiv
\left| 11100 \right\rangle
$$

will be understood as spins 1, 2 and 3 up and spins 4 and 5 down.

The Hilbert space of the composite system is simply the tensor product of the single-particle Hilbert spaces on the chain. An arbitrary basis state will be denoted by

$$
\left| \sigma \right\rangle
=
\left| \sigma_{1} \sigma_{2} \cdots \sigma_{N} \right\rangle
$$

where each $\sigma_{i} \in \{1,0\}$.

The spin operators for each particle are constant multiples of the Pauli matrices:

$$
\left( S_{x}, S_{y}, S_{z} \right)
=
\dfrac{\hbar}{2}
\left( \sigma_{x}, \sigma_{y}, \sigma_{z} \right)
$$

We will assume $\hbar = 1$ as stated before. Taking $J=1$ as stated before, the Hamiltonian is given by

$$
H =
\sum_{i=1}^{N-1}
\mathbf{S}_{i} \cdot \mathbf{S}_{i+1}
=
\sum_{i=1}^{N-1} h_{i}
$$

where

$$
h_{i} := \mathbf{S}_{i} \cdot \mathbf{S}_{i+1}
$$

which may be called the $\emph{local interaction term}$.

$$
h_{i}
=
\mathbf{S}_{i} \cdot \mathbf{S}_{i+1}
=
S_i^x S_{i+1}^x
+
S_i^y S_{i+1}^y
+
S_i^z S_{i+1}^z
$$

Working with the raising and lowering operators $S^{\pm}$ is more straightforward than $S_x$ and $S_y$ because their action on the computational basis states is known. We have

$$
S^{\pm} = S^{x} \pm i S^{y}
$$

From which, we get

$$
S^{x}_{j}
=
\dfrac{1}{2}
\left(
S^{+}_{j} + S^{-}_{j}
\right)
\qquad , \qquad
S^{y}_{j}
=
\dfrac{1}{2i}
\left(
S^{+}_{j} - S^{-}_{j}
\right)
$$

Substitution and simplification will give

$$
h_{i}
=
\dfrac{1}{2}
\left(
S^{+}_{i} S^{-}_{i+1}
+
S^{-}_{i} S^{+}_{i+1}
\right)
+
S^{z}_{i} S^{z}_{i+1}
$$

The action of the raising and lowering operators is as follows:

\begin{align}\notag
&S^{+} \left| 0 \right\rangle = \left| 1 \right\rangle
\qquad &&, \qquad
S^{+} \left| 1 \right\rangle = 0
\\ \notag
&S^{-} \left| 0 \right\rangle = 0
\qquad &&, \qquad
S^{-} \left| 1 \right\rangle = \left| 0 \right\rangle
\end{align}

How does $h_i$ act on an arbitrary basis state?

$$
h_{i}
\left| \cdots \sigma_{i} \sigma_{i+1} \cdots \right\rangle
=
\left(
\dfrac{1}{2}
\left(
S^{+}_{i} S^{-}_{i+1}
+
S^{-}_{i} S^{+}_{i+1}
\right)
+
S^{z}_{i} S^{z}_{i+1}
\right)
\left| \cdots \sigma_{i} \sigma_{i+1} \cdots \right\rangle
$$

If the spins $i$ and $i+1$ are aligned, i.e., they are both either up or down, we get

$$
h_{i}
\left| \cdots \sigma_{i} \sigma_{i+1} \cdots \right\rangle
=
\dfrac{1}{4}
\left| \cdots \sigma_{i} \sigma_{i+1} \cdots \right\rangle
$$

And if they have opposite orientations, we get

$$
h_{i}
\left| \cdots \sigma_{i} \sigma_{i+1} \cdots \right\rangle
=
-\dfrac{1}{4}
\left| \cdots \sigma_{i} \sigma_{i+1} \cdots \right\rangle
+
\dfrac{1}{2}
\left| \cdots \sigma_{i+1} \sigma_{i} \cdots \right\rangle
$$

In the second ket, the spins for sites $i$ and $i+1$ get switched.

Let us consider a simple case of $N=4$ for the sake of illustration and suppose that we want to find the action of $H$ on the given basis state $\left|1010\right\rangle$.

\begin{align} \notag
H\left|1010\right\rangle
&= \sum_{i=1}^{3}h_i\left|1010\right\rangle \\ \notag
&= h_1\left|1010\right\rangle
+ h_2\left|1010\right\rangle
+ h_3\left|1010\right\rangle \\ \notag
&= -\dfrac{1}{4}\left|1010\right\rangle
+ \dfrac{1}{2}\left|0110\right\rangle \\ \notag
&\quad -\dfrac{1}{4}\left|1010\right\rangle
+ \dfrac{1}{2}\left|1100\right\rangle \\ \notag
&\quad -\dfrac{1}{4}\left|1010\right\rangle
+ \dfrac{1}{2}\left|1001\right\rangle
\end{align}

And hence

$$
H\left|1010\right\rangle
=
-\dfrac{3}{4}\left|1010\right\rangle
+
\dfrac{1}{2}
\left(
\left|0110\right\rangle
+
\left|1100\right\rangle
+
\left|1001\right\rangle
\right)
$$

By finding the action of $H$ on this basis state, we can determine the nonzero elements of the column of $H$ corresponding to $\left|1010\right\rangle$.

Now our objective is to find the matrix representation of a Hamiltonian for a chain of $N$ spins. We will attack the problem by evaluating the effect of $H$ on an arbitrary basis state $\left|\sigma\right\rangle$. The implementation below therefore takes a basis state $\left|\sigma\right\rangle$ as input and returns the nonzero terms and their coefficients in the resulting linear combination $H\left|\sigma\right\rangle$.

Reproducing this process for every basis state gives the complete matrix representation of $H$.

In [35]:
# This cell is to implement the action of the Heisenberg Hamiltonian on an arbitrary basis state |\sigma>.
# The Hamiltonian is given by:
# H = \sum_{i = 1}^{N-1} h_i
# Where
# h_i = S_i \cdot S_{i+1}.
# The action of h_i on arbitrary states has been computed by hand and the result will be implemented here.


import numpy as np

# Some default inputs for N and |\sigma>.
sigma = '1010'

# The basis state should be given as a binary string; '1001...' is to be
# interpreted as up, down, down, up, ....
# We will convert it to a tuple for further manipulation.
def convert_to_tuple(sigma):
    return tuple(int(x) for x in sigma)

# We will store the basis states as decimal integers.
# This function converts a binary basis state to decimal:
def binary_to_decimal(number):
    # The input should be a tuple
    N = len(number)
    powers = [2**(N-i) for i in range(1, N+1)]
    decimal = np.array(number) * np.array(powers)
    return np.sum(decimal)

# H |\sigma> = \sum_{i} h_i |\sigma>.
def apply_H(sigma, n_spins):
    # Store the basis states and their coefficients in a dictionary
    # representing the resulting linear combination.
    lc = {}

    # Convert the basis-state string to a tuple.
    sigma = convert_to_tuple(sigma)

    # Iterate over the nearest-neighbor pairs.
    for i in range(1, n_spins):

        # Depending on sigma[i-1] and sigma[i], the action of h_i is different.
        # If the basis already exists in lc, just add the value
        # else, create the key entry and store the value as well.
        if sigma[i-1] == sigma[i]:
            basis = int(binary_to_decimal(sigma))
            coefficient = 0.25

            if basis in lc:
                lc[basis] += coefficient
            else:
                lc[basis] = coefficient

        else:
            basis_1 = int(binary_to_decimal(sigma))
            coefficient_1 = -0.25

            if basis_1 in lc:
                lc[basis_1] += coefficient_1
            else:
                lc[basis_1] = coefficient_1

            basis_2 = list(sigma)
            basis_2[i-1], basis_2[i] = basis_2[i], basis_2[i-1]
            basis_2 = tuple(basis_2)
            basis_2 = int(binary_to_decimal(basis_2))
            coefficient_2 = 0.5

            if basis_2 in lc:
                lc[basis_2] += coefficient_2
            else:
                lc[basis_2] = coefficient_2

    return lc

In [36]:
# A quick check
print(apply_H(sigma, 4))

{10: -0.75, 6: 0.5, 12: 0.5, 9: 0.5}


### 2.2 Construction and exact diagonalization for a chain of 4 spins

### Constructing the Hamiltonian Matrix

Now that we have determined the action of the Hamiltonian on an arbitrary basis state, we can construct its complete matrix representation. We will first do this using a dense matrix for a small system with $N=4$ spins. This allows us to inspect the matrix directly and verify that our implementation agrees with the results obtained analytically.

The rows and columns of the matrix are indexed by the integers

$$
0,1,\ldots,2^N-1,
$$

which correspond to the computational-basis states written in their binary representation. We construct the matrix column by column; for each column index, we convert the corresponding integer into an $N$-bit binary string, use `apply_H` to evaluate the action of $H$ on that basis state, and then use the resulting dictionary to fill the nonzero entries of the column.

Once the matrix has been constructed, we can diagonalize it numerically to obtain the energy spectrum and the corresponding eigenstates. In particular, the lowest eigenvalue gives the ground-state energy, while its eigenvector gives the ground-state wavefunction in the computational basis.

After verifying the construction for this small system, we will examine why storing the Hamiltonian as a dense matrix quickly becomes impractical as $N$ increases, despite the Hamiltonian being highly sparse. This will motivate the use of sparse matrix representations.

In [37]:
# The rows and columns of H are indexed by the integers
# 0, 1, 2, ..., 2^N - 1. We iterate over the columns and
# apply H to each corresponding basis state. Since apply_H
# takes a binary-string representation of the state, we first
# convert the integer index to an N-bit binary string.

# Specify N:
N = 4

# Initialize H with entries all equal to zero:
H = np.zeros((2**N, 2**N))

# The outer loop iterates over columns, while the inner loop
# fills the nonzero entries of each column.
for i in range(2**N):
    # First, we need to convert the matrix indices from decimals to binaries 
    # because our apply_H function gets entries of that form.
    state = format(i, f'0{N}b')

    # Now that we have the state, we can act apply_H on it and get the
    # desired dictionary to fill the corresponding column.
    lc = apply_H(state, N)

    for key in lc.keys():
        H[key, i] = lc[key]

# Let's see how many nonzero elements H has
print('Number of nonzero elements in H :', np.count_nonzero(H))
print('Total numbe rof elements in H :', 4**N)

# Some quick checks
print(np.allclose(H, H.T))
print(H[:, 10])

# Now let us diagonalize this matrix directly using the numpy linalg module
eigenvalues, eigenvectors = np.linalg.eigh(H)
print(eigenvalues)
E_0 = eigenvalues[0]
psi_0 = eigenvectors[:, 0]
print(np.allclose(H @ psi_0, eigenvalues[0] * psi_0))
print("Ground-state energy:\n", E_0)
print("Ground-state vector:\n", psi_0)

Number of nonzero elements in H : 40
Total numbe rof elements in H : 256
True
[ 0.    0.    0.    0.    0.    0.    0.5   0.    0.    0.5  -0.75  0.
  0.5   0.    0.    0.  ]
[-1.6160254  -0.95710678 -0.95710678 -0.95710678 -0.25       -0.25
 -0.25        0.1160254   0.45710678  0.45710678  0.45710678  0.75
  0.75        0.75        0.75        0.75      ]
True
Ground-state energy:
 -1.6160254037844386
Ground-state vector:
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.49429245e-01
  0.00000000e+00 -5.57677536e-01  4.08248290e-01  0.00000000e+00
  0.00000000e+00  4.08248290e-01 -5.57677536e-01  1.29787663e-18
  1.49429245e-01 -5.26713922e-19  1.41132570e-19  0.00000000e+00]


The construction above gives the complete Hamiltonian matrix for the $N=4$ chain. We can then diagonalize the matrix directly to obtain the full energy spectrum and the corresponding eigenstates. In particular, the lowest eigenvalue gives the ground-state energy, while its corresponding eigenvector gives the ground-state wavefunction in the computational basis.

For $N=4$, the Hilbert space has dimension

$$
d=2^N=16,
$$

so the Hamiltonian is a $16\times16$ matrix. This is small enough that storing and diagonalizing the dense matrix is straightforward. However, the Hilbert-space dimension grows exponentially with the number of spins, while the Hamiltonian itself remains highly sparse. A dense representation therefore stores a very large number of entries that are identically zero.

For a system of $N$ spins, a dense representation requires

$$
(2^N)^2 = 4^N
$$

matrix elements. In contrast, each column of the Heisenberg Hamiltonian contains at most $N$ nonzero elements. It is therefore natural to exploit this sparsity when studying larger systems.

We will next construct the same Hamiltonian using a sparse matrix representation. This will allow us to work with larger chains and will also prepare us for using sparse eigensolvers to obtain the ground state without computing the entire spectrum.

In [38]:
# We will now take a leap towards a more optimized and efficient way of
# storing sparse Hamiltonians. We will use the COO construction from 
# scipy.sparse module.

# Define a function that gets the required list for a COO construction
def coo_lists(apply_H, N):
    """
    :Parameters:
    apply_H : the function that computes the action of H on an arbitrary basis state
    N : number of spins on the chain

    :Returns:
    rows, columns, values for the COO construction
    """
    # Initialize rows, columns and values
    rows, columns, values = [], [], []

    # We will iterate on columns and store the required entries for our lists
    for i in range(2**N):
        # Convert the column index to binary first
        state = format(i, f'0{N}b')

        # Get the dictionary corresponding to this column
        lc = apply_H(state, N)

        # Now store the entries
        for key, value in lc.items():
            rows.append(key)
            columns.append(i)
            values.append(value)

    return rows, columns, values

In [39]:
# A quick check
rows, columns, values = coo_lists(apply_H, N)
print(len(rows))
print(len(columns))
print(len(values))

40
40
40


In [40]:
# Now moving on to the sparse construction for a chain of size N = 4
import scipy as sp

# Create the sparse Hamiltonian
rows, columns, values = coo_lists(apply_H, N)
H_coo = sp.sparse.coo_matrix((values, (rows, columns)), shape = (2**N, 2**N))

# Quick checks
print(H_coo.nnz)
print(np.allclose(H_coo.toarray(), H))

# COO is excellent for constructing the matrix but for operations such as
# multplication or H∣ψ⟩, CSR is a more suitable construction.
# So we will create a CSR version of our Hamiltonian.
H_csr = H_coo.tocsr()

# Verification
print(H_csr.nnz)
print(np.allclose(H_csr.toarray(), H))

40
True
40
True


### 2.3 Using `eigsh` for sparse matrices

In the previous section, we constructed the Hamiltonian as a sparse matrix and converted it to the CSR format. We can now take advantage of this representation when diagonalizing the Hamiltonian. Rather than computing the entire spectrum, as we did with `numpy.linalg.eigh`, we are primarily interested in the ground state, which corresponds to the lowest eigenvalue. SciPy's `eigsh` function allows us to compute only a selected number of eigenpairs of a real symmetric or complex Hermitian sparse matrix, making it particularly useful for this purpose.

In [41]:
# Find the ground-state eigenvalue and eigenvector using a sparse eigensolver
eigenvalues, eigenvectors = sp.sparse.linalg.eigsh(H_csr, k=1, which='SA')
print(eigenvalues)
print(eigenvectors)
print(np.allclose(eigenvalues[0], E_0))
psi_0_sparse = eigenvectors[:, 0]
print(np.allclose(H_csr @ psi_0_sparse,
                  eigenvalues[0] * psi_0_sparse))

[-1.6160254]
[[-5.55111512e-17]
 [-2.77555756e-17]
 [-4.16333634e-17]
 [ 1.49429245e-01]
 [ 5.55111512e-17]
 [-5.57677536e-01]
 [ 4.08248290e-01]
 [ 5.55111512e-17]
 [-2.77555756e-17]
 [ 4.08248290e-01]
 [-5.57677536e-01]
 [-5.55111512e-17]
 [ 1.49429245e-01]
 [-2.77555756e-17]
 [ 5.55111512e-17]
 [-2.77555756e-17]]
True
True


### 2.4 Summary and limitations of exact diagonalization

We have constructed the Hamiltonian of the Heisenberg chain directly from its action on the computational basis. For a chain of $N$ spins, the Hilbert-space dimension is $2^{N}$, and although the Hamiltonian is sparse due to the local nature of the interactions, the dimension of the Hilbert space still grows exponentially with system size.

For small systems, exact diagonalization provides direct access to the energy spectrum and eigenstates and therefore serves as an important benchmark for approximate methods. In particular, we can use the exact ground-state energy and wavefunction to assess the accuracy of methods such as DMRG and neural quantum states.

However, the exponential growth of the Hilbert space makes exact diagonalization impractical beyond relatively small system sizes. This motivates the use of methods that can represent or approximate many-body wavefunctions without explicitly storing the full $2^{N}$-dimensional state vector.

## 3. Variational Monte Carlo

The exact diagonalization approach used in the previous section provides an exact solution for sufficiently small systems, but its computational cost grows exponentially with system size. We therefore need alternative approaches that can approximate the ground state without explicitly representing the full many-body wavefunction.

Variational Monte Carlo (VMC) provides one such approach. Rather than solving the eigenvalue problem directly, we introduce a parameterized trial wavefunction and optimize its parameters to minimize the expectation value of the energy.

The basic idea is:

$$
\text{choose a trial wavefunction}
\;\longrightarrow\;
\text{estimate its energy}
\;\longrightarrow\;
\text{optimize its parameters}.
$$

The first step is to understand the variational principle on which this approach is based.

### 3.1 The variational principle

Suppose that the Hamiltonian has eigenstates

$$
H|n\rangle = E_n|n\rangle,
$$

with the ground-state energy satisfying

$$
E_0 \leq E_1 \leq E_2 \leq \cdots.
$$

Consider an arbitrary normalized trial state $|\psi\rangle$. We can expand it in the complete set of energy eigenstates:

$$
|\psi\rangle = \sum_n c_n |n\rangle,
$$

where normalization requires

$$
\sum_n |c_n|^2 = 1.
$$

The expectation value of the energy is

$$
E[\psi]
=
\langle\psi|H|\psi\rangle.
$$

Substituting the expansion of $|\psi\rangle$ gives

$$
E[\psi]
=
\sum_n |c_n|^2 E_n.
$$

Since every eigenenergy satisfies $E_n \geq E_0$,

$$
E[\psi]
\geq
E_0\sum_n |c_n|^2
=
E_0.
$$

Therefore,

$$
\boxed{E[\psi]\geq E_0.}
$$

This is the variational principle: the expectation value of the energy of any normalized trial state provides an upper bound to the exact ground-state energy.

### 3.2 Parameterized trial wavefunctions

In practice, we consider a family of trial wavefunctions parameterized by a set of parameters $\theta$:

$$
|\psi_\theta\rangle.
$$

The corresponding variational energy is

$$
E(\theta)
=
\frac{
\langle\psi_\theta|H|\psi_\theta\rangle
}{
\langle\psi_\theta|\psi_\theta\rangle
}.
$$

The variational principle guarantees that

$$
E(\theta)\geq E_0
$$

for every choice of $\theta$. Therefore, our goal is to find the parameters that minimize the variational energy:

$$
\boxed{
\min_\theta E(\theta).
}
$$

The quality of the resulting approximation depends on the expressive power of the chosen family of trial wavefunctions. In later sections, we will consider two different types of variational representations: matrix product states and neural quantum states.

### 3.3 Monte Carlo estimation of the variational energy

The variational energy can be written in terms of the coefficients of the trial wavefunction as

$$
E(\theta)
=
\dfrac{
\sum_{\sigma,\sigma'}
\psi_\theta^*(\sigma')
H_{\sigma',\sigma}
\psi_\theta(\sigma)
}{
\sum_\sigma |\psi_\theta(\sigma)|^2
}.
$$

The double sum in the numerator and the sum in the denominator both become exponentially expensive as the size of the Hilbert space increases. We therefore want to rewrite the expression in a form that can be estimated by sampling basis states.

We define the local energy of a basis state $\sigma$ as

$$
E_{\mathrm{loc}}(\sigma)
=
\sum_{\sigma'}
H_{\sigma',\sigma}
\frac{\psi_\theta(\sigma')}{\psi_\theta(\sigma)},
$$

where we initially consider real-valued trial wavefunctions.

Using this definition, the variational energy becomes

$$
E(\theta)
=
\frac{
\sum_\sigma |\psi_\theta(\sigma)|^2 E_{\mathrm{loc}}(\sigma)
}{
\sum_\sigma |\psi_\theta(\sigma)|^2
}.
$$

This suggests defining

$$
p_\theta(\sigma)
=
\frac{|\psi_\theta(\sigma)|^2}
{\sum_{\sigma'}|\psi_\theta(\sigma')|^2},
$$

which can be interpreted as a probability distribution over the computational-basis states. The energy can then be written as

$$
E(\theta)
=
\sum_\sigma p_\theta(\sigma)E_{\mathrm{loc}}(\sigma).
$$

Therefore, instead of explicitly summing over all basis states, we can estimate the energy by drawing samples from $p_\theta(\sigma)$ and averaging their local energies.

Before implementing the sampling procedure, we first consider a simple trial wavefunction for which the expectation value can be evaluated directly.

In [42]:
# The trial wavefuncrion is
# |psi> = 1/sqrt(6) (|0000> + |0011> + |0101> + |1010> + |1100> + |1111>).

E_avg = []
configs = ['0011', '0101', '0110', '1001', '1010', '1100']
for config in configs:
    lc = apply_H(config, n_spins = 4)
    H_elements = [lc[key] for key in lc]
    E_loc = sum(H_elements)
    E_avg.append(E_loc)

E = (1 / len(E_avg)) * sum(E_avg)

print(E)

0.75


In this test, we used a uniform trial wavefunction over the six basis states in the $S_z=0$ sector. Since all amplitudes are equal, the ratio

$$
\frac{\psi(\sigma')}{\psi(\sigma)}
$$

is equal to one. Therefore, the local energy reduces to the sum of the nonzero Hamiltonian matrix elements generated by `apply_H`.

The resulting energy is therefore the direct average of the local energies over the six configurations. This gives a simple test of the connection between the Hamiltonian action implemented in the exact diagonalization section and the local-energy expression used in VMC.

### 3.4 Constructing a Monte Carlo move

The expression

$$
E(\theta)
=
\sum_\sigma p_\theta(\sigma)E_{\mathrm{loc}}(\sigma)
$$

is useful only if we can obtain configurations distributed according to $p_\theta(\sigma)$.

We therefore need a procedure that moves through the space of computational-basis states and eventually generates samples from this distribution.

For the Heisenberg chain, we choose to work within a fixed $S_z$ sector. The Monte Carlo move should therefore preserve the total $S_z$. A simple way to achieve this is to choose one up spin and one down spin and exchange their positions.

For a state such as

$$
|\sigma\rangle = |1001011101\rangle,
$$

the move selects one position containing $1$ and one position containing $0$ and exchanges them. The number of up and down spins is unchanged, and therefore the total $S_z$ is preserved.

We can now turn this procedure into a function so that it can be used repeatedly during the Monte Carlo sampling.

In [43]:
# We will now work out a function that takes a basis state
# as input and returns another basis state in the same S_{z}
# sector (since total S_{z} must be conserved for this Hamiltonian).

def propose_move(sigma):
    # Choose two random spins (one up and the other one, down)
    # and then exchange them. This way, the total S_{z} is conserved.
    # We will give the input state as a string and store the indices
    # of 1's and 0's in two separate lists and then, we select a random
    # index from each list and swap their corresponding spins.

    proposal = sigma

    # Iterate over the string and store the indices on 1's and 0's
    ones = [i for i, char in enumerate(proposal) if char == '1']
    zeros = [i for i, char in enumerate(proposal) if char == '0']
    # print(ones, zeros)

    # Select one random index from each list
    i, j = np.random.choice(ones), np.random.choice(zeros)

    # Swap the selected spins
    proposal = list(proposal)
    proposal[i], proposal[j] = proposal[j], proposal[i]
    proposal = ''.join(proposal)

    return proposal

### 3.5 Checking the symmetry constraint

Since the proposed move exchanges one up spin with one down spin, it should preserve the total $S_z$ of the state.

For a spin-$1/2$ basis state,

$$
S_z^{\mathrm{tot}}
=
\frac{1}{2}(N_\uparrow-N_\downarrow).
$$

Since

$$
N_\downarrow=N-N_\uparrow,
$$

this can also be written as

$$
S_z^{\mathrm{tot}}
=
N_\uparrow-\frac{N}{2}.
$$

We can use this expression to check that repeated Monte Carlo moves remain within the same $S_z$ sector.

In [44]:
# This function will compute the total S_{z} for an
# arbitrary basis state.

def S_z(sigma):
    # The basis state is passed into the function as
    # a string of binary number.

    # The total S_{z} is equal to (1/2)*(N_up - N_down)
    # S_{z} = (1/2)*(N_up - (N - N_up)) = N_up - N/2.
    # Get the number of up spins
    N_up = sigma.count('1')
    S_z = N_up - 0.5*len(sigma)

    return S_z

In [45]:
# Let's do a sanity check to make sure our proposed_move function
# does not take states outside of its corresponding sector.

# Some arbitrary input basis state
sigma = '111001101111'

# Number of samples
n_samples = 10

# Create a list to store the proposals
sigmas = [sigma]
for i in range(n_samples):
    sigma = propose_move(sigma)
    sigmas.append(sigma)

# Check if they all have the same S_{z}
S_z_list = [S_z(state) for state in sigmas]

print(S_z_list)

[3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0]


### 3.6 Metropolis-Hastings sampling

We now need a way to generate configurations distributed according to

$$
p_\theta(\sigma)
=
\frac{|\psi_\theta(\sigma)|^2}
{\sum_{\sigma'}|\psi_\theta(\sigma')|^2}.
$$

For a small system, we could calculate this distribution explicitly by evaluating the wavefunction for every basis state. However, this is not feasible for large systems because the number of basis states grows exponentially with $N$.

The idea behind Monte Carlo sampling is to construct a Markov chain of basis states

$$
\sigma_0 \rightarrow \sigma_1 \rightarrow \sigma_2
\rightarrow \cdots
$$

whose stationary distribution is $p_\theta(\sigma)$. If the chain has reached this stationary distribution, the configurations visited by the chain can be used as samples from $p_\theta$.

To construct such a chain, suppose that the current configuration is $\sigma$. We first propose a new configuration $\sigma'$ according to some proposal distribution

$$
q(\sigma'|\sigma).
$$

The proposed configuration is not always accepted. Instead, we accept it with a probability chosen such that the resulting Markov chain has $p_\theta(\sigma)$ as its stationary distribution. This is the role of the Metropolis-Hastings acceptance rule.

The general acceptance probability is

$$
A(\sigma\rightarrow\sigma')
=
\min\left[
1,
\frac{
p_\theta(\sigma')q(\sigma|\sigma')
}{
p_\theta(\sigma)q(\sigma'|\sigma)
}
\right].
$$

In our case, the proposal consists of exchanging one up spin with one down spin. This proposal is symmetric: the probability of proposing $\sigma'$ from $\sigma$ is the same as the probability of proposing $\sigma$ from $\sigma'$,

$$
q(\sigma'|\sigma)=q(\sigma|\sigma').
$$

Therefore, the proposal probabilities cancel:

$$
A(\sigma\rightarrow\sigma')
=
\min\left[
1,
\frac{p_\theta(\sigma')}{p_\theta(\sigma)}
\right].
$$

The target distribution itself contains an unknown normalization factor. However, this factor is the same for both configurations, so it also cancels in the ratio:

$$
\frac{p_\theta(\sigma')}{p_\theta(\sigma)}
=
\frac{|\psi_\theta(\sigma')|^2}
{|\psi_\theta(\sigma)|^2}.
$$

Thus, the acceptance probability becomes

$$
A(\sigma\rightarrow\sigma')
=
\min\left[
1,
\frac{|\psi_\theta(\sigma')|^2}
{|\psi_\theta(\sigma)|^2}
\right].
$$

This means that we never need to calculate the normalization of the wavefunction. We only need to evaluate its relative weights for the current and proposed configurations.

In [46]:
# A function that implements bra multiplication to find the
# coefficient \psi(\sigma) of the basis state |\sigma> in the
# linear combination of the trial wavefunction |\psi> given by
# \sum_{\sigma} \psi(\sigma) |\sigma>.
# The trial wavefunction is stored as a dictionary that contains
# basis states as keys and their coefficients as their corresponding values.

def bra(trial_psi, basis):
    # trial_psi is the given trial wavefunction and
    # basis (given as a binary string), is the basis
    # whose coefficient we want to get. The idea is to
    # go over the the keys of the dictionary and 
    # capture the value for the key that is equal
    # to our basis state.

    # psi is the coefficient corresponding to basis
    # which we initialize as 0 and change it if the
    # basis is indeed present in the linear combination.
    psi = 0

    if basis in trial_psi:
        psi = trial_psi[basis]

    return psi

In [47]:
# Quick check
trial_psi = {
    '0011': 0.10,
    '0101': 0.30,
    '0110': -0.20,
    '1001': 0.40,
    '1010': 0.50,
    '1100': -0.60
}
print(bra(trial_psi, '1010'))
print(bra(trial_psi, '1100'))
print(bra(trial_psi, '1111'))

0.5
-0.6
0


For a given pair of configurations, we can now calculate the acceptance probability using the ratio of their wavefunction amplitudes.

In [48]:
# In this cell, a function acc_prob is defined that calculates
# the acceptance probability for the Metropolis_hastings algorithm.
# The acceptance probability is given by
# A (\sigma \rightarrow \sigma') = min[1, R]
# where R = \dfrac{|\psi(\sigma')|^2}{|\psi(\sigma)|^2}.

def acc_prob(trial_psi, sigma, sigma_prime):
    # The basis states are given as binary strings
    # and the ratio R is computed and finally, the
    # acceptance probability is computed and returned as
    # output of the function. The trial wavefunction
    # is given as a dictionary as before.

    # Get the coefficients
    psi_sigma, psi_sigma_prime = bra(trial_psi, sigma), bra(trial_psi, sigma_prime)

    # Compute R
    R = (psi_sigma_prime / psi_sigma) ** 2

    # Compute A
    A = min(1, R)

    return A

In [49]:
# Test
trial_psi = {
    '0011': 0.10,
    '0101': 0.30,
    '0110': -0.20,
    '1001': 0.40,
    '1010': 0.50,
    '1100': -0.60
}
sigma        = '1010'
sigma_prime  = '0101'

print(acc_prob(trial_psi, sigma, sigma_prime))
print(acc_prob(trial_psi, sigma_prime, sigma))

0.36
1


### 3.7 Implementing the Metropolis sampler

We can now combine the proposal move and the acceptance rule into a complete Metropolis-Hastings sampler.

At each step, the algorithm:

1. starts from the current configuration $\sigma$;
2. proposes a new configuration $\sigma'$;
3. calculates the acceptance probability $A(\sigma\rightarrow\sigma')$;
4. accepts the proposed configuration with probability $A$;
5. otherwise keeps the current configuration.

The resulting sequence of configurations forms a Markov chain. After an initial equilibration period, the chain can be used to estimate expectation values with respect to $p_\theta(\sigma)$.

We keep the initial part of the chain separately as burn-in samples and use the remaining configurations as the production samples.

In [50]:
# The metropolis_sampler is defined to sample from
# a distribution p(\sigma) when its normalization factor is
# not known or it's difficult to evaluate. The trial
# wavefunction is passed to the function and some random
# initial \sigma is given as the starting point and then,
# moves are proposed and they are either accepted or rejected.
# In the meantime, the samples are stored and after excluding
# the burn-ins, we hope to have obtained a Markov chain that
# has the stationary distribution p(\sigma).

import numpy as np
np.random.seed(24)

def metropolis_sampler(psi, N, n, initial_state = 'random'):
    """
    :Parameters:
    psi: the trial wavefunction (dictionary)
    N: number of Metropolis iteration (integer)
    n: number of burn-in samples to discard (integer)

    :returns:
    burn_ins: the list of burn_ins (list)
    samples: the list of samples obtained using the Metropolis-Hastings algorithm (list)
    acceptance_rate: The portion of accepted samples
    """

    # Initialize the list to store the samples
    samples = []

    # Select some basis state at random if initial_state = 'random'
    if initial_state == 'random':
        sigma = np.random.choice(list(psi.keys()))
    # Otherwise, take the basis state given as the input
    else:
        sigma = initial_state

    # Record the number of accepted samples
    N_acc = 0

    # A loop will implement the sampling
    for i in range(N):
        sigma_prime = propose_move(sigma)

        # Now accept/reject
        u = np.random.uniform(0, 1)
        A = acc_prob(psi, sigma, sigma_prime)

        if u <= A:
            # The move is accepted with probability = A
            samples.append(sigma_prime)
            sigma = sigma_prime
            N_acc += 1

        else:
            # The move is rejected
            samples.append(sigma)

    # give the burn-ins and the samples as separate lists
    burn_ins, samples = samples[0:n], samples[n:]

    # Fraction of proposed moves that were accepted
    acceptance_rate = N_acc/N

    return burn_ins, samples, acceptance_rate

In [51]:
# This function calculates the probability of obtaining sigma
# from the samples distribution p(sigma)
def probability(samples, sigma):
    samples = np.array(samples)

    return np.sum(samples == sigma) / len(samples)

In [52]:
# Test the sampler

trial_psi = {
    '0011': 0.10,
    '0101': 0.30,
    '0110': -0.20,
    '1001': 0.40,
    '1010': 0.50,
    '1100': -0.60
}
burn_ins, samples, acceptance_rate = metropolis_sampler(
    trial_psi, N=10000, n=1000
)

states = list(trial_psi.keys())
p_tot = 0
for sigma in states:
    print(f'{sigma}, {probability(samples, sigma):.4f}')
    p_tot += probability(samples, sigma)

print(f'{p_tot:.4f}')
print(f"Acceptance rate: {acceptance_rate:.4f}")

0011, 0.0130
0101, 0.1009
0110, 0.0436
1001, 0.1821
1010, 0.2661
1100, 0.3943
1.0000
Acceptance rate: 0.5019


### 3.8 Computing the local energy

The Metropolis sampler gives us configurations distributed according to

$$
p_\theta(\sigma)\propto|\psi_\theta(\sigma)|^2.
$$

For each sampled configuration, we need to calculate its local energy

$$
E_{\mathrm{loc}}(\sigma)
=
\sum_{\sigma'}
H_{\sigma',\sigma}
\frac{\psi_\theta(\sigma')}{\psi_\theta(\sigma)}.
$$

The function `apply_H` developed in the exact diagonalization section already gives us the nonzero elements $H_{\sigma',\sigma}$ for a given basis state. We can therefore use it directly to evaluate the local energy without constructing the full Hamiltonian matrix.

In [53]:
# e_loc is a function that calculates the local energy 
# E_{loc}(\sigma) = \sum_{\sigma'} H_{\sigma', \sigma} \dfrac{\psi(\sigma')}{\psi(\sigma)}.
# Takes the trial wavefunction and a basis state as inputs
# and computes E_{loc}.
def e_loc(trial_psi, basis):
    # Initialize the e_loc value
    e_loc = 0
    n_spins = len(basis)

    # Use apply_H to get the required elements of H.
    lc = apply_H(basis, n_spins)

    # Sum over the nonzero terms generated by H |sigma>.
    for key in lc:
        # Get the coefficients
        key_binary = format(key, f'0{n_spins}b')
        psi_sigma, psi_sigma_prime = bra(trial_psi, basis), bra(trial_psi, key_binary)

        # Compute the contribution to the local energy.
        r = (psi_sigma_prime / psi_sigma) * lc[key]
        e_loc += r

    return e_loc

In [54]:
# Get a quick check on the function
trial_psi = {
    '0011': 0.10,
    '0101': 0.30,
    '0110': -0.20,
    '1001': 0.40,
    '1010': 0.50,
    '1100': -0.60
}
e_loc(trial_psi, '1010')

-1.15

### 3.9 Monte Carlo estimation of the energy

Once the configurations have been sampled and the local energy has been evaluated, the variational energy can be estimated as a simple sample average:

$$
E(\theta)
=
\sum_\sigma p_\theta(\sigma)E_{\mathrm{loc}}(\sigma)
\approx
\frac{1}{M}
\sum_{m=1}^{M}
E_{\mathrm{loc}}(\sigma_m),
$$

where $\{\sigma_m\}$ are configurations obtained from the Metropolis-Hastings chain.

We can therefore estimate the expectation value of the Hamiltonian without explicitly summing over the entire Hilbert space.

In [55]:
# To estimate the expectation value of H in the state |\psi>,
# we use the Monte Carlo estimator
# E = (1/M) \sum_{m} E_{loc}(\sigma_m).

def monte_carlo_energy(trial_psi, n_samples, burn_ins):
    # Given the trial wavefunction, compute the expectation
    # value of energy in this state using n_samples.
    _, samples, _ = metropolis_sampler(trial_psi, n_samples, burn_ins)
    energies = [e_loc(trial_psi, basis) for basis in samples]
    energies = np.array(energies)
    e_expected = np.mean(energies)

    return e_expected

In [56]:
# Let's see what the Monte Carlo method estimates as the
# expectation value of energy for this trial wavefunction
print(f'{monte_carlo_energy(trial_psi, 10000, 1000):.6f}')

-0.350952


### 3.10 Comparing with the exact expectation value

For the small $N=4$ system, we can still evaluate the expectation value directly because the number of basis states in the chosen sector is small.

Using

$$
E
=
\frac{
\sum_\sigma |\psi(\sigma)|^2 E_{\mathrm{loc}}(\sigma)
}{
\sum_\sigma |\psi(\sigma)|^2
},
$$

we can calculate the exact expectation value for the chosen trial wavefunction and compare it with the Monte Carlo estimate.

This provides a useful check that the sampling and local-energy implementations are working correctly.

In [57]:
# Calculate the exact energy for the trial wavefunction
# and compare with the Monte Carlo approxiamtion.

def exact_energy(trial_psi):
    # E_exact = (1 / normalization) \times \sum_{\sigma} |\psi(\sigma)|^2 E_{loc}(\sigma)
    # where normalization = \sum_{\sigma} |\psi(\sigma)|^2
    normalization = 0
    energy = 0
    for key in trial_psi:
        energy += (trial_psi[key] ** 2) * e_loc(trial_psi, key)
        normalization += trial_psi[key] ** 2

    e_exact = energy / normalization

    return e_exact

In [58]:
# Test the exact energy for a small system (N = 4)
trial_psi = {
    '0011': 0.10,
    '0101': 0.30,
    '0110': -0.20,
    '1001': 0.40,
    '1010': 0.50,
    '1100': -0.60
}
print(f'{exact_energy(trial_psi):.6f}')

-0.354396


### 3.11 Statistical convergence of the Monte Carlo estimate

The Monte Carlo estimate is not exactly equal to the expectation value because it is obtained from a finite number of samples.

We therefore expect the result to fluctuate between different runs. Increasing the number of samples should generally reduce the statistical uncertainty, although the samples generated by a Markov chain are correlated and are therefore not equivalent to independent samples.

Because both the number of samples and the number of burn-in steps are varied simultaneously, this experiment is not intended to determine an optimal choice of either parameter. Rather, it illustrates the statistical fluctuations of the Monte Carlo estimate for different sampling conditions.

In [59]:
# Run a few tests and compare the exact values with the
# Monte Carlo approximation.

trial_psi = {
    '0011': 0.10,
    '0101': 0.30,
    '0110': -0.20,
    '1001': 0.40,
    '1010': 0.50,
    '1100': -0.60
}

n_samples_list = [2500, 5000, 10000, 25000, 50000]
burn_ins_list = [100, 250, 500, 750, 1000]

for n_samples in n_samples_list:
    for burn_ins in burn_ins_list:
        exact = exact_energy(trial_psi)
        approx = monte_carlo_energy(trial_psi, n_samples, burn_ins)

        # absolute relative error percentage = |[(monte carlo - exact) / exact]| * 100
        abs_rel_error = abs((approx/exact - 1)) * 100
        print(f'{n_samples:>6} samples and {burn_ins:<4} burn-ins: {abs_rel_error:>7.3f}%')

  2500 samples and 100  burn-ins:   0.135%
  2500 samples and 250  burn-ins:  17.073%
  2500 samples and 500  burn-ins:   0.704%
  2500 samples and 750  burn-ins:   1.133%
  2500 samples and 1000 burn-ins:   9.223%
  5000 samples and 100  burn-ins:   6.155%
  5000 samples and 250  burn-ins:   0.089%
  5000 samples and 500  burn-ins:   3.153%
  5000 samples and 750  burn-ins:   5.433%
  5000 samples and 1000 burn-ins:   0.287%
 10000 samples and 100  burn-ins:   1.528%
 10000 samples and 250  burn-ins:   5.843%
 10000 samples and 500  burn-ins:   6.417%
 10000 samples and 750  burn-ins:   0.886%
 10000 samples and 1000 burn-ins:   2.243%
 25000 samples and 100  burn-ins:   0.882%
 25000 samples and 250  burn-ins:   0.328%
 25000 samples and 500  burn-ins:   1.140%
 25000 samples and 750  burn-ins:   0.921%
 25000 samples and 1000 burn-ins:   2.647%
 50000 samples and 100  burn-ins:   0.114%
 50000 samples and 250  burn-ins:   1.055%
 50000 samples and 500  burn-ins:   1.203%
 50000 samp

### 3.12 Burn-in and equilibration

The Metropolis-Hastings chain is initialized from some arbitrary configuration. In general, this initial configuration is not distributed according to the desired stationary distribution.

The early part of the chain can therefore depend on the initial condition. These initial samples are referred to as burn-in or equilibration samples and are discarded before estimating physical quantities.

The purpose of burn-in is not to make the samples independent. Rather, it is to reduce the influence of the initial condition on the subsequent samples.

For our small $N=4$ system, the target distribution is known exactly, so we can use it to investigate this behavior directly.

We first fix the initial configuration and increase the length of the Markov chain. The empirical distribution can then be compared with the exact distribution.

This experiment illustrates that the empirical probabilities fluctuate for short chains and become more stable as more samples are collected.

In [60]:
# We will compare the values of the empirical probabilities (obtained from the samples)
# with the exact probability computed analytically from trial_psi.
def exact_prob(trial_psi, state):
    # The probability of being at a cetain basis state is given by
    # |\psi(state)|^2 divided by the normalization factor
    norm_factor = [trial_psi[state] ** 2 for state in trial_psi]
    norm_factor = sum(norm_factor)

    return (trial_psi[state] ** 2) / norm_factor

# The next function, will compute the empirical probability of a certain state
# by dividing the number of occurrences of that state by the total number of samples.
def empirical_prob(sample, state):
    sample = np.array(sample)

    return np.mean(sample == state)

In [61]:
# We want to get a better understanding of the origin of burn-in samples
# and how the chain behaves as we start off from a random basis state and
# collect the proposals. We will investigate this phenomena for N = 4 spins.

# Use a trial wavefunction
trial_psi = {
    '0011': 0.10,
    '0101': 0.30,
    '0110': -0.20,
    '1001': 0.40,
    '1010': 0.50,
    '1100': -0.60
}

# Now run a test for a number of sample sizes and compare the probabilities
sample_sizes = [10, 100, 500, 1000, 5000, 10000]

# Iterate and collect data with no burn-ins
print(f"{'Sample size':<12}{'0011':<10}{'0101':<10}{'0110':<10}{'1001':<10}{'1010':<10}{'1100':<10}")

# Take some initial state as basis for analysis
initial = '0101'
for sample_size in sample_sizes:
    _, sample, _ = metropolis_sampler(trial_psi, N = sample_size, n = 0, initial_state = initial)
    row = ""
    for state in trial_psi:
        row += f'{empirical_prob(sample, state):<10f}'
    print(f"{sample_size:<12}{row:<10}")

# The final row is for the exact values
final_row = ""
for state in trial_psi:
    final_row += f'{exact_prob(trial_psi, state):<10f}'
print(f"{'Exact':<12}{final_row}")

# By comparing rows, we can see that our empirical distribution aproaches
# the exact distribution with errors that seem to be decreasing as the 
# size of the sample increases.

Sample size 0011      0101      0110      1001      1010      1100      
10          0.000000  0.200000  0.000000  0.400000  0.100000  0.300000  
100         0.000000  0.090000  0.050000  0.150000  0.250000  0.460000  
500         0.016000  0.122000  0.038000  0.174000  0.230000  0.420000  
1000        0.013000  0.096000  0.048000  0.178000  0.221000  0.444000  
5000        0.010000  0.100800  0.049200  0.172000  0.265000  0.403000  
10000       0.011700  0.102000  0.046300  0.178100  0.267500  0.394400  
Exact       0.010989  0.098901  0.043956  0.175824  0.274725  0.395604  


The empirical probabilities approach the exact probabilities as the number of samples increases, although the convergence is not monotonic because the Monte Carlo estimates fluctuate from run to run.

### 3.13 Investigating the effect of burn-in

To investigate the effect of burn-in separately from the randomness between different Monte Carlo runs, we generate one long Markov chain and analyze the same trajectory after discarding different numbers of initial samples.

For a chosen burn-in $b$, the production samples are

$$
\{\sigma_b,\sigma_{b+1},\ldots,\sigma_{M}\}.
$$

We can then compare the empirical distribution obtained after different values of $b$ with the exact distribution.

For this small system, the effect of changing the burn-in is relatively small compared with the statistical fluctuations of the Monte Carlo estimate. This indicates that the initial transient is short compared with the length of the chain for the particular trial wavefunction and proposal used here.

The experiment also illustrates why burn-in should not simply be chosen as the value that gives the smallest error with respect to the exact answer. Burn-in is intended to reduce dependence on the initial condition, whereas statistical fluctuations remain even after equilibration.

In [62]:
# Now let's investigate the burn-ins further. How do they affect
# the convergence and how many initial samples should be discarded
# as burn-ins to get a better empirical sample?

# Generate the long chain first
burn_ins, samples, acceptance_rate = metropolis_sampler(
    trial_psi,
    N=10000,
    n=0,
    initial_state='0101'
)

# Various values for the number of discarded samples
burn_in_values = [0, 10, 50, 100, 250, 500, 1000]

# Compare the empirical and exact values for each state and each value of burn-in
for burn_in in burn_in_values:
    production = samples[burn_in:]

    print(f"\nBurn-in = {burn_in}")
    for state in trial_psi:
        print(f"{state}: {empirical_prob(production, state):.6f}", f"{exact_prob(trial_psi, state):.6f}")



Burn-in = 0
0011: 0.011600 0.010989
0101: 0.106200 0.098901
0110: 0.045800 0.043956
1001: 0.176200 0.175824
1010: 0.269800 0.274725
1100: 0.390400 0.395604

Burn-in = 10
0011: 0.011612 0.010989
0101: 0.106206 0.098901
0110: 0.045846 0.043956
1001: 0.176276 0.175824
1010: 0.269570 0.274725
1100: 0.390490 0.395604

Burn-in = 50
0011: 0.011658 0.010989
0101: 0.106332 0.098901
0110: 0.046030 0.043956
1001: 0.176080 0.175824
1010: 0.269849 0.274725
1100: 0.390050 0.395604

Burn-in = 100
0011: 0.011616 0.010989
0101: 0.106162 0.098901
0110: 0.045758 0.043956
1001: 0.176566 0.175824
1010: 0.270606 0.274725
1100: 0.389293 0.395604

Burn-in = 250
0011: 0.011590 0.010989
0101: 0.106872 0.098901
0110: 0.046051 0.043956
1001: 0.176205 0.175824
1010: 0.270359 0.274725
1100: 0.388923 0.395604

Burn-in = 500
0011: 0.011579 0.010989
0101: 0.106842 0.098901
0110: 0.045895 0.043956
1001: 0.175895 0.175824
1010: 0.270211 0.274725
1100: 0.389579 0.395604

Burn-in = 1000
0011: 0.011333 0.010989
0101: 0.10

### 3.14 Summary

In this section, we introduced the main ideas behind variational Monte Carlo and implemented the basic sampling machinery needed for later neural quantum states.

Starting from a parameterized trial wavefunction, we rewrote the variational energy as an expectation value over the probability distribution

$$
p_\theta(\sigma)\propto|\psi_\theta(\sigma)|^2.
$$

We then used the Metropolis-Hastings algorithm to sample configurations from this distribution. The Hamiltonian action implemented in the exact diagonalization section was used to calculate the local energy for each sampled configuration, allowing the variational energy to be estimated as a Monte Carlo average.

For the small $N=4$ system, we compared the Monte Carlo results with exact calculations and investigated the effects of sample size and burn-in. These tests provide a basic validation of the VMC machinery.

The main components developed here will be reused when we introduce neural quantum states, where the wavefunction is no longer given explicitly but is instead represented by a parameterized neural network.

In [65]:
# We have computed the truncated Schmidt decomposition and
# have the following normalized state:
# |\psi> = 1/sqrt(5) (|0000> + |0011> + |0101> + |1100> + |1111>).
# We want to compute the expectation value for energy at this state
# which is <\psi | H | \psi> and then comapre it with the value of
# the ground state energy obtained from ED which was E_0 = -1.616025.

import numpy as np

# Pass the truncated wavefunction as a dictionary.
truncated_psi = {
    '0000': 1/np.sqrt(5),
    '0011': 1/np.sqrt(5),
    '0101': 1/np.sqrt(5),
    '1100': 1/np.sqrt(5),
    '1111': 1/np.sqrt(5)
}

E_truncated = exact_energy(truncated_psi)

print(E_truncated)

0.45


## 4.1 Motivation: Representing Many-Body Wavefunctions

In the previous sections, we studied the ground state of the spin-½ antiferromagnetic Heisenberg XXX chain using exact diagonalization (ED) and introduced the basic machinery of variational Monte Carlo (VMC). Exact diagonalization provides direct access to the Hamiltonian and its eigenstates, while VMC allows us to estimate the energy of a trial wavefunction through Monte Carlo sampling. However, both approaches leave us with an important question: how should we represent a many-body wavefunction as the system size increases?

For a chain of $N$ spin-½ particles, the Hilbert-space dimension is

$$
\dim\mathcal H=2^N.
$$

A general many-body wavefunction can be written as

$$
|\psi\rangle
=
\sum_{\sigma_1,\ldots,\sigma_N}
\psi_{\sigma_1\ldots\sigma_N}
|\sigma_1,\ldots,\sigma_N\rangle,
$$

where $\sigma_i\in\{0,1\}$ labels the local spin configuration at site $i$.

The coefficient tensor $\psi_{\sigma_1\ldots\sigma_N}$ therefore contains $2^N$ complex amplitudes. As $N$ increases, explicitly storing a generic wavefunction quickly becomes impractical.

We have already encountered one way to reduce the size of the problem: exploiting symmetries. For example, restricting a chain of $N=16$ spins to the sector with total magnetization $S^z_{\mathrm{tot}}=0$ gives a Hilbert-space dimension of

$$
\dim\mathcal H_{S^z_{\mathrm{tot}}=0}
=
\binom{16}{8}
=
12870.
$$

This restriction is possible because total magnetization zero requires eight spins to point up and eight to point down. Although the symmetry restriction makes the problem more manageable, the dimension of the sector still grows rapidly with system size.

The central idea behind alternative wavefunction representations is to exploit structure in physically relevant quantum states. Instead of storing every coefficient independently, we seek a representation built from smaller objects that captures the important correlations between different parts of the system.

For one-dimensional systems, one of the most important such representations is the **Matrix Product State (MPS)**.

The MPS representation does not eliminate the exponential complexity of representing arbitrary quantum states. Its advantage is that many physically relevant low-energy states can be represented accurately with relatively small bond dimensions, making the number of required parameters much smaller than the dimension of the full Hilbert space.

To understand why this is possible, we will begin by constructing the MPS representation and then derive its connection to the singular value decomposition (SVD), Schmidt decomposition, and entanglement.

## 4.2 Matrix Product States

The idea behind a Matrix Product State is to factorize these amplitudes into a product of smaller tensors, introducing auxiliary indices that connect neighboring sites.

For an open-boundary chain, an MPS represents the wavefunction as

$$
|\psi\rangle
=
\sum_{\sigma_1,\ldots,\sigma_N}
\psi_{\sigma_1\ldots\sigma_N}
|\sigma_1,\ldots,\sigma_N\rangle,
$$

where the coefficients are written as

$$
\psi_{\sigma_1\ldots\sigma_N}
=
\sum_{\alpha_1,\ldots,\alpha_{N-1}}
A^{\sigma_1}_{\alpha_1}
A^{\sigma_2}_{\alpha_1\alpha_2}
\cdots
A^{\sigma_N}_{\alpha_{N-1}}.
$$

Here, $\sigma_i$ is the physical index at site $i$, while $\alpha_i$ is a virtual index connecting the tensors at sites $i$ and $i+1$. The dimension of a virtual index is called the **bond dimension**:

$$
\chi_i=\dim(\alpha_i).
$$

For a local Hilbert-space dimension $d$, a bulk MPS tensor has dimensions

$$
\chi_{i-1}\times d\times\chi_i,
$$

with $\chi_0=\chi_N=1$ for open boundary conditions.

For a spin-½ chain, $d=2$. If all bond dimensions are bounded by a common value $\chi$, each bulk tensor contains approximately $d\chi^2$ entries. Since there are $N-2$ bulk tensors, together with two boundary tensors containing approximately $d\chi$ entries each, the total number of entries scales as

$$
O(Nd\chi^2).
$$

Thus, for fixed $\chi$, the number of parameters grows linearly with the system size rather than exponentially. This is the main computational motivation for using MPS.

But what does the bond dimension actually control? It determines how many independent components can connect the degrees of freedom on opposite sides of a bond. A larger $\chi$ allows a more expressive representation, whereas a smaller $\chi$ restricts the correlations that can be represented across that bond.

We will make this statement precise by connecting the MPS representation to the singular value decomposition and the Schmidt decomposition. This will also explain why bond dimension is closely related to entanglement.

## 4.3 Schmidt Decomposition from the Singular Value Decomposition

To understand the meaning of the bond dimension, let us consider a bipartition of the chain into two subsystems: the left subsystem, containing sites $1,\ldots,\ell$, and the right subsystem, containing sites $\ell+1,\ldots,N$.

We can group the basis states of each subsystem and rewrite the wavefunction as

$$
|\psi\rangle
=
\sum_{i,j} C_{ij}|i\rangle_L|j\rangle_R,
$$

where $i$ labels the basis states of the left subsystem, $j$ labels those of the right subsystem, and $C$ is a matrix containing the wavefunction coefficients. This matrix is obtained simply by reshaping the original coefficient tensor.

We can now apply the singular value decomposition (SVD):

$$
C=U\Sigma V^\dagger.
$$

In components, this reads

$$
C_{ij}
=
\sum_{\alpha}U_{i\alpha}s_\alpha V^\dagger_{\alpha j},
$$

where $s_\alpha$ are the singular values of $C$. Substituting this expression into the wavefunction gives

$$
|\psi\rangle
=
\sum_{i,j,\alpha}
U_{i\alpha}s_\alpha V^\dagger_{\alpha j}
|i\rangle_L|j\rangle_R.
$$

Rearranging the sums, we obtain

$$
|\psi\rangle
=
\sum_\alpha s_\alpha
\left(\sum_i U_{i\alpha}|i\rangle_L\right)
\left(\sum_j V^\dagger_{\alpha j}|j\rangle_R\right).
$$

This expression suggests defining new states for the two subsystems:

$$
|\alpha_L\rangle
=
\sum_i U_{i\alpha}|i\rangle_L,
$$

$$
|\alpha_R\rangle
=
\sum_j V^\dagger_{\alpha j}|j\rangle_R.
$$

Because the columns of $U$ and $V$ are orthonormal, these states form orthonormal sets. The wavefunction therefore takes the form

$$
|\psi\rangle
=
\sum_\alpha s_\alpha
|\alpha_L\rangle|\alpha_R\rangle.
$$

This is the **Schmidt decomposition** of the state across the chosen bipartition. The singular values $s_\alpha$ are its Schmidt coefficients.

To see how normalization constrains these coefficients, we calculate the norm of the state:

$$
\begin{aligned}
\langle\psi|\psi\rangle
&=
\sum_{\alpha,\beta}
s_\alpha s_\beta
\langle\alpha_L|\beta_L\rangle
\langle\alpha_R|\beta_R\rangle\\
&=
\sum_{\alpha,\beta}
s_\alpha s_\beta
\delta_{\alpha\beta}\delta_{\alpha\beta}\\
&=
\sum_\alpha s_\alpha^2.
\end{aligned}
$$

Here, we have used the orthonormality of the Schmidt states and the fact that singular values are nonnegative real numbers. For a normalized state, $\langle\psi|\psi\rangle=1$, and therefore

$$
\sum_\alpha s_\alpha^2=1.
$$

The important point is that the SVD has transformed the coefficient matrix into a sum of products of orthonormal states on the left and right, with each product weighted by a single coefficient. The number of nonzero Schmidt coefficients is called the **Schmidt rank** of the bipartition.

We will now use this result to understand what the MPS bond dimension measures and why it determines how much entanglement a given representation can capture.

## 4.4 Understanding the Singular Value Decomposition

Before continuing with MPS, it is useful to understand more carefully what the singular value decomposition is doing. In the previous section, we used the SVD to obtain the Schmidt decomposition, but we did not yet examine in detail how the singular values and singular vectors are related to the original matrix.

Consider a general complex matrix

$$
C\in\mathbb C^{m\times n}.
$$

Its singular value decomposition is

$$
C=U\Sigma V^\dagger,
$$

where $U$ and $V$ are unitary matrices and $\Sigma$ is a diagonal matrix containing the singular values

$$
s_1,s_2,\ldots,s_r,
$$

with $s_\alpha\geq0$.

A useful way to understand the SVD is through the two matrices

$$
CC^\dagger
$$

and

$$
C^\dagger C.
$$

These matrices are generally different. If $C$ is an $m\times n$ matrix, then

$$
CC^\dagger\in\mathbb C^{m\times m},
$$

whereas

$$
C^\dagger C\in\mathbb C^{n\times n}.
$$

Thus, they act on different vector spaces. The first acts on the space associated with the rows of $C$, while the second acts on the space associated with the columns of $C$.

Both matrices are Hermitian and positive semidefinite. For any vector $|v\rangle$,

$$
\langle v|C^\dagger C|v\rangle
=
\langle Cv|Cv\rangle
=
\|C|v\rangle\|^2
\geq0.
$$

Similarly,

$$
\langle u|CC^\dagger|u\rangle
=
\|C^\dagger|u\rangle\|^2
\geq0.
$$

Therefore, both matrices have real, nonnegative eigenvalues.

Let us first consider an eigenvector of $C^\dagger C$:

$$
C^\dagger C|v_\alpha\rangle
=
\lambda_\alpha|v_\alpha\rangle.
$$

Multiplying this equation from the left by $C$ gives

$$
CC^\dagger C|v_\alpha\rangle
=
\lambda_\alpha C|v_\alpha\rangle.
$$

Therefore, whenever $\lambda_\alpha>0$, the vector $C|v_\alpha\rangle$ is an eigenvector of $CC^\dagger$ with the same eigenvalue $\lambda_\alpha$.

We can normalize this vector. Its norm is

$$
\begin{aligned}
\|C|v_\alpha\rangle\|^2
&=
\langle v_\alpha|C^\dagger C|v_\alpha\rangle\\
&=
\lambda_\alpha
\langle v_\alpha|v_\alpha\rangle\\
&=
\lambda_\alpha,
\end{aligned}
$$

where we have assumed that $|v_\alpha\rangle$ is normalized. Therefore,

$$
|u_\alpha\rangle
=
\frac{C|v_\alpha\rangle}{\sqrt{\lambda_\alpha}}
$$

is a normalized eigenvector of $CC^\dagger$.

Equivalently,

$$
C|v_\alpha\rangle
=
\sqrt{\lambda_\alpha}|u_\alpha\rangle.
$$

We can also obtain the reverse relation. Starting from

$$
CC^\dagger|u_\alpha\rangle
=
\lambda_\alpha|u_\alpha\rangle
$$

and multiplying by $C^\dagger$ gives

$$
C^\dagger CC^\dagger|u_\alpha\rangle
=
\lambda_\alpha C^\dagger|u_\alpha\rangle.
$$

Thus, for $\lambda_\alpha>0$,

$$
C^\dagger|u_\alpha\rangle
=
\sqrt{\lambda_\alpha}|v_\alpha\rangle.
$$

These relations show that the nonzero eigenvalues of $CC^\dagger$ and $C^\dagger C$ are the same. The two matrices can have different numbers of zero eigenvalues because they can have different dimensions, but their nonzero spectra coincide.

The singular values of $C$ are defined as the positive square roots of these nonzero eigenvalues:

$$
s_\alpha=\sqrt{\lambda_\alpha}.
$$

Therefore,

$$
s_\alpha^2=\lambda_\alpha.
$$

Using the orthonormal eigenvectors of $CC^\dagger$ and $C^\dagger C$, we can write the matrix as

$$
C
=
\sum_{\alpha=1}^{r}
s_\alpha
|u_\alpha\rangle
\langle v_\alpha|,
$$

where

$$
r=\operatorname{rank}(C)
$$

is the number of nonzero singular values.

In matrix notation, this is precisely

$$
C=U\Sigma V^\dagger.
$$

The vectors $|v_\alpha\rangle$ are therefore the eigenvectors of $C^\dagger C$, while the vectors $|u_\alpha\rangle$ are the eigenvectors of $CC^\dagger$. The singular values connect the two sets through

$$
C|v_\alpha\rangle=s_\alpha|u_\alpha\rangle
$$

and

$$
C^\dagger|u_\alpha\rangle=s_\alpha|v_\alpha\rangle.
$$

This is the mathematical structure underlying the SVD.

### A simple example

Consider the matrix

$$
C=
\frac{1}{\sqrt{10}}
\begin{pmatrix}
2&1\\
1&2
\end{pmatrix}.
$$

Since this matrix is real and symmetric,

$$
C^\dagger=C.
$$

In this particular example, we therefore have

$$
CC^\dagger=C^\dagger C=C^2.
$$

This equality is a special property of this matrix and should not be assumed for a general matrix.

We calculate

$$
C^\dagger C
=
\frac{1}{10}
\begin{pmatrix}
2&1\\
1&2
\end{pmatrix}
\begin{pmatrix}
2&1\\
1&2
\end{pmatrix}
=
\frac{1}{10}
\begin{pmatrix}
5&4\\
4&5
\end{pmatrix}.
$$

The eigenvalues of this matrix are

$$
\lambda_1=\frac{9}{10},
\qquad
\lambda_2=\frac{1}{10}.
$$

Therefore, the singular values are

$$
s_1=\frac{3}{\sqrt{10}},
\qquad
s_2=\frac{1}{\sqrt{10}}.
$$

The corresponding normalized eigenvectors of $C^\dagger C$ are

$$
|v_1\rangle
=
\frac{1}{\sqrt2}
\begin{pmatrix}
1\\
1
\end{pmatrix},
\qquad
|v_2\rangle
=
\frac{1}{\sqrt2}
\begin{pmatrix}
1\\
-1
\end{pmatrix}.
$$

These are the right singular vectors.

We can obtain the left singular vectors using

$$
|u_\alpha\rangle
=
\frac{C|v_\alpha\rangle}{s_\alpha}.
$$

For the first vector,

$$
C|v_1\rangle
=
\frac{3}{\sqrt{10}}|v_1\rangle,
$$

so

$$
|u_1\rangle=|v_1\rangle.
$$

Similarly,

$$
C|v_2\rangle
=
\frac{1}{\sqrt{10}}|v_2\rangle,
$$

and therefore

$$
|u_2\rangle=|v_2\rangle.
$$

Thus, for this particular example,

$$
U=V=
\frac{1}{\sqrt2}
\begin{pmatrix}
1&1\\
1&-1
\end{pmatrix},
$$

and

$$
\Sigma=
\begin{pmatrix}
\frac{3}{\sqrt{10}}&0\\
0&\frac{1}{\sqrt{10}}
\end{pmatrix}.
$$

The original matrix can then be reconstructed as

$$
C=U\Sigma V^\dagger.
$$

This example illustrates the complete procedure: we obtain the right singular vectors from the eigenvalue problem for $C^\dagger C$, obtain the singular values from its eigenvalues, and then obtain the left singular vectors from the action of $C$ on the right singular vectors.

The important point for our application to many-body wavefunctions is that the SVD is therefore not simply an arbitrary factorization. It identifies orthogonal directions in the two subsystems that are connected by the matrix $C$, together with the strength $s_\alpha$ of each connection. In the Schmidt decomposition, these directions become the Schmidt states and the singular values become the Schmidt coefficients.

This is what allows the SVD to provide a natural way of determining which components of a many-body wavefunction are most important when constructing an MPS.

## 4.5 Constructing an MPS through Successive SVDs

So far, we have shown that the SVD of a coefficient matrix gives the Schmidt decomposition across a chosen bipartition, and that the Schmidt rank determines the minimum bond dimension required for an exact representation across that cut. We can now use the same idea repeatedly to factorize the full coefficient tensor into an MPS.

Consider the wavefunction

$$
|\psi\rangle
=
\sum_{\sigma_1,\ldots,\sigma_N}
\psi_{\sigma_1\ldots\sigma_N}
|\sigma_1,\ldots,\sigma_N\rangle.
$$

We begin by grouping the first physical index $\sigma_1$ into the row index and all remaining indices into the column index. This reshapes the coefficient tensor into a matrix:

$$
C_{\sigma_1,(\sigma_2\ldots\sigma_N)}
=
\psi_{\sigma_1\ldots\sigma_N}.
$$

Applying the SVD gives

$$
C=U\Sigma V^\dagger.
$$

We can identify the first MPS tensor with the left singular vectors:

$$
A^{\sigma_1}_{\alpha_1}
=
U_{\sigma_1,\alpha_1}.
$$

The remaining factors form a new tensor,

$$
T_{\alpha_1,\sigma_2,\ldots,\sigma_N}
=
(\Sigma V^\dagger)_{\alpha_1,(\sigma_2\ldots\sigma_N)}.
$$

Substituting these definitions into the original coefficient tensor yields

$$
\psi_{\sigma_1\ldots\sigma_N}
=
\sum_{\alpha_1}
A^{\sigma_1}_{\alpha_1}
T_{\alpha_1,\sigma_2,\ldots,\sigma_N}.
$$

We have separated the first site from the rest of the chain. The remaining tensor $T$ still contains all the information associated with sites $2,\ldots,N$, together with the bond index $\alpha_1$.

We can repeat the procedure on this remaining tensor. Group the indices $(\alpha_1,\sigma_2)$ into a row index and $(\sigma_3,\ldots,\sigma_N)$ into a column index, then reshape it into a matrix:

$$
M_{(\alpha_1,\sigma_2),(\sigma_3\ldots\sigma_N)}
=
T_{\alpha_1,\sigma_2,\ldots,\sigma_N}.
$$

Applying another SVD gives

$$
M=U^{[2]}\Sigma^{[2]}V^{[2]\dagger}.
$$

The second MPS tensor is obtained by reshaping the left singular vectors:

$$
A^{\sigma_2}_{\alpha_1\alpha_2}
=
U^{[2]}_{(\alpha_1,\sigma_2),\alpha_2}.
$$

The remaining factors define a new residual tensor that contains the degrees of freedom from sites $3,\ldots,N$ and the new bond index $\alpha_2$.

Repeating this process site by site, we obtain the factorization

$$
\psi_{\sigma_1\ldots\sigma_N}
=
\sum_{\alpha_1,\ldots,\alpha_{N-1}}
A^{\sigma_1}_{\alpha_1}
A^{\sigma_2}_{\alpha_1\alpha_2}
\cdots
A^{\sigma_N}_{\alpha_{N-1}}.
$$

This is precisely the open-boundary MPS representation introduced in Section 4.2.

If we retain all nonzero singular values at every step, the procedure gives an exact representation of the original wavefunction. If we retain only the largest $\chi_i$ singular values at a given step, the corresponding bond dimension is reduced and the representation becomes approximate.

The important distinction is that **successive SVDs provide the factorization itself, while truncation controls the bond dimensions and introduces approximation**. This distinction will become especially important when we discuss DMRG, where SVD is used to manage the bond dimension during optimization.

Although this construction shows that any finite-chain wavefunction can be represented as an MPS with sufficiently large bond dimensions, it does not yet explain how to optimize an MPS efficiently. To understand that, we first need to examine a freedom inherent in the representation: different sets of tensors can describe exactly the same physical state.

## 4.5 Bond Dimension and Entanglement

Let us return to the MPS representation and consider the bond connecting sites $\ell$ and $\ell+1$. This bond divides the chain into a left subsystem containing sites $1,\ldots,\ell$ and a right subsystem containing sites $\ell+1,\ldots,N$.

Grouping the physical indices on either side of the bond, we can write the wavefunction coefficients as

$$
C_{ij}
=
\sum_{\alpha=1}^{\chi_\ell}
L_{i\alpha}R_{\alpha j},
$$

where $L$ and $R$ collect the tensors on the left and right sides of the bond, respectively. In matrix notation, this is simply

$$
C=LR.
$$

The shared index $\alpha$ takes at most $\chi_\ell$ values. Consequently,

$$
\operatorname{rank}(C)\leq\chi_\ell.
$$

The rank of $C$ is precisely the number of nonzero Schmidt coefficients. Denoting this number by $r_\ell$, we have

$$
r_\ell=\operatorname{rank}(C)
=\#\{\alpha:s_\alpha>0\}.
$$

It follows that an exact MPS representation must satisfy

$$
\chi_\ell\geq r_\ell.
$$

We have therefore obtained a precise interpretation of the bond dimension: **it determines the maximum number of independent Schmidt components that an MPS can represent across a bond.** If the bond dimension is smaller than the Schmidt rank of the state, an exact representation is impossible.

In practice, however, retaining every Schmidt component is often unnecessary. Consider the Schmidt decomposition

$$
|\psi\rangle
=
\sum_{\alpha=1}^{r_\ell}
s_\alpha|\alpha_L\rangle|\alpha_R\rangle,
$$

where the Schmidt coefficients are arranged in descending order,

$$
s_1\geq s_2\geq\cdots\geq s_{r_\ell}>0.
$$

We can construct an approximation by retaining only the largest $\chi$ coefficients:

$$
|\tilde{\psi}\rangle
=
\sum_{\alpha=1}^{\chi}
s_\alpha|\alpha_L\rangle|\alpha_R\rangle.
$$

Because the Schmidt states are orthonormal, the squared norm of this truncated state is

$$
\langle\tilde{\psi}|\tilde{\psi}\rangle
=
\sum_{\alpha=1}^{\chi}s_\alpha^2.
$$

For a normalized original state, the Schmidt coefficients satisfy $\sum_{\alpha=1}^{r_\ell}s_\alpha^2=1$. The discarded weight is therefore

$$
\epsilon_{\mathrm{disc}}
=
\sum_{\alpha=\chi+1}^{r_\ell}s_\alpha^2
=
1-\langle\tilde{\psi}|\tilde{\psi}\rangle.
$$

The discarded weight measures how much of the original state's squared norm lies in the omitted Schmidt components. Notice that the truncated state $|\tilde{\psi}\rangle$ is not generally normalized.

Why should we retain the largest Schmidt coefficients? The SVD has an important optimality property: among matrices of rank at most $\chi$, retaining the largest $\chi$ singular values gives the best approximation to $C$ in the Frobenius norm. SVD truncation therefore provides a principled way to reduce the bond dimension while controlling the approximation error.

### Bond Dimension and Entanglement Entropy

The Schmidt coefficients also tell us how entanglement is distributed across a bipartition. To see this, consider the density matrix of the full system,

$$
\rho=|\psi\rangle\langle\psi|.
$$

If we want to describe the left subsystem alone, we must trace out the degrees of freedom belonging to the right subsystem. This gives the reduced density matrix

$$
\rho_L=\operatorname{Tr}_R(\rho).
$$

Substituting the Schmidt decomposition into the full density matrix yields

$$
\rho
=
\sum_{\alpha,\beta}
s_\alpha s_\beta
|\alpha_L\rangle\langle\beta_L|
\otimes
|\alpha_R\rangle\langle\beta_R|.
$$

Taking the partial trace over the right subsystem and using the orthonormality of the Schmidt states,

$$
\operatorname{Tr}_R
\left(
|\alpha_R\rangle\langle\beta_R|
\right)
=
\langle\beta_R|\alpha_R\rangle
=
\delta_{\alpha\beta},
$$

we obtain

$$
\rho_L
=
\sum_\alpha s_\alpha^2
|\alpha_L\rangle\langle\alpha_L|.
$$

Thus, the Schmidt states $|\alpha_L\rangle$ are eigenvectors of the reduced density matrix, and the corresponding eigenvalues are

$$
\lambda_\alpha=s_\alpha^2.
$$

The entanglement entropy between the two subsystems is defined as the von Neumann entropy of the reduced density matrix:

$$
S_L=-\operatorname{Tr}(\rho_L\log\rho_L).
$$

Using its eigenvalues, we can express the entropy directly in terms of the Schmidt coefficients:

$$
S_L=-\sum_\alpha s_\alpha^2\log(s_\alpha^2),
$$

where terms with $s_\alpha=0$ are understood to contribute zero.

This expression reveals that entanglement depends not only on the number of nonzero Schmidt coefficients but also on how their weights are distributed. For example, a state with one nonzero Schmidt coefficient has zero entanglement entropy across the cut, whereas a state with several equally weighted Schmidt coefficients has a larger entropy.

For a normalized state with Schmidt rank $r_\ell$, the entropy satisfies

$$
0\leq S_L\leq\log r_\ell.
$$

Since an MPS with bond dimension $\chi_\ell$ has Schmidt rank at most $\chi_\ell$, its entanglement entropy across that bond is bounded by

$$
S_L\leq\log\chi_\ell.
$$

This is a fundamental connection between the structure of an MPS and the quantum state it can represent. A larger bond dimension allows more Schmidt components and can accommodate greater entanglement across the corresponding cut. However, increasing the bond dimension does not necessarily increase the entanglement of a particular state; it only increases the representation's capacity to capture it.

We have now connected four important ideas: the SVD identifies the Schmidt components, the Schmidt rank determines the minimum bond dimension required for an exact representation, truncation allows us to trade accuracy for a smaller representation, and the Schmidt coefficients determine the entanglement entropy.

The natural next question is how to construct an MPS for an entire wavefunction using successive SVDs, rather than considering only a single bipartition.

## 4.6 Constructing an MPS through Successive SVDs

We can now use the SVD repeatedly to construct an MPS representation of an entire many-body wavefunction. The procedure is simply an extension of the bipartite decomposition we considered earlier.

Consider the wavefunction

$$
|\psi\rangle
=
\sum_{\sigma_1,\ldots,\sigma_N}
\psi_{\sigma_1\ldots\sigma_N}
|\sigma_1,\ldots,\sigma_N\rangle.
$$

We begin by grouping the first physical index $\sigma_1$ into the row index and all remaining indices into the column index. This reshapes the coefficient tensor into a matrix:

$$
C_{\sigma_1,(\sigma_2\ldots\sigma_N)}
=
\psi_{\sigma_1\ldots\sigma_N}.
$$

We can then perform an SVD:

$$
C=U\Sigma V^\dagger.
$$

The left singular vectors depend only on $\sigma_1$, so they can be identified with the first MPS tensor:

$$
A^{\sigma_1}_{\alpha_1}
=
U_{\sigma_1,\alpha_1}.
$$

The remaining factors can be combined into a new tensor,

$$
T_{\alpha_1,\sigma_2,\ldots,\sigma_N}
=
(\Sigma V^\dagger)_{\alpha_1,(\sigma_2\ldots\sigma_N)}.
$$

Therefore,

$$
\psi_{\sigma_1\ldots\sigma_N}
=
\sum_{\alpha_1}
A^{\sigma_1}_{\alpha_1}
T_{\alpha_1,\sigma_2,\ldots,\sigma_N}.
$$

At this point, the first site has been separated from the rest of the chain, while the remaining tensor still contains all information about sites $2,\ldots,N$.

We now repeat the same procedure for the second site. We group the indices $(\alpha_1,\sigma_2)$ into a row index and $(\sigma_3,\ldots,\sigma_N)$ into a column index:

$$
M_{(\alpha_1,\sigma_2),(\sigma_3\ldots\sigma_N)}
=
T_{\alpha_1,\sigma_2,\ldots,\sigma_N}.
$$

Applying the SVD gives

$$
M=U^{[2]}\Sigma^{[2]}V^{[2]\dagger}.
$$

The second MPS tensor is obtained by reshaping the left singular vectors:

$$
A^{\sigma_2}_{\alpha_1\alpha_2}
=
U^{[2]}_{(\alpha_1,\sigma_2),\alpha_2}.
$$

The remaining factors are combined into another residual tensor:

$$
T'_{\alpha_2,\sigma_3,\ldots,\sigma_N}
=
(\Sigma^{[2]}V^{[2]\dagger})_{\alpha_2,(\sigma_3\ldots\sigma_N)}.
$$

The wavefunction can therefore be written as

$$
\psi_{\sigma_1\ldots\sigma_N}
=
\sum_{\alpha_1,\alpha_2}
A^{\sigma_1}_{\alpha_1}
A^{\sigma_2}_{\alpha_1\alpha_2}
T'_{\alpha_2,\sigma_3,\ldots,\sigma_N}.
$$

We continue this procedure site by site. At each step, the indices already generated on the left are combined with the physical index of the next site, while all remaining physical indices are grouped into the other side of the matrix.

After repeating the procedure through the chain, we obtain

$$
\psi_{\sigma_1\ldots\sigma_N}
=
\sum_{\alpha_1,\ldots,\alpha_{N-1}}
A^{\sigma_1}_{\alpha_1}
A^{\sigma_2}_{\alpha_1\alpha_2}
\cdots
A^{\sigma_{N-1}}_{\alpha_{N-2}\alpha_{N-1}}
A^{\sigma_N}_{\alpha_{N-1}}.
$$

Substituting this expression into the wavefunction gives the MPS representation

$$
|\psi\rangle
=
\sum_{\sigma_1,\ldots,\sigma_N}
\sum_{\alpha_1,\ldots,\alpha_{N-1}}
A^{\sigma_1}_{\alpha_1}
A^{\sigma_2}_{\alpha_1\alpha_2}
\cdots
A^{\sigma_N}_{\alpha_{N-1}}
|\sigma_1,\ldots,\sigma_N\rangle.
$$

At each SVD, the number of retained singular values determines the dimension of the newly introduced virtual index. Therefore, the bond dimension $\chi_i$ at each bond is directly related to the Schmidt rank across that bipartition.

If all nonzero singular values are retained at every step, the resulting MPS represents the original wavefunction exactly. If only the largest $\chi_i$ singular values are retained, the corresponding bond dimension is reduced and the representation becomes approximate.

This also makes clear why the SVD is so central to MPS: it simultaneously provides the factorization needed to construct the tensors and the singular-value spectrum that tells us how much information can be retained when the bond dimension is truncated.

We will now work through explicit small-system examples of this procedure, writing down the wavefunction and MPS tensors step by step.

## 4.7 Worked Examples: From SVD to MPS Construction

We can now apply the SVD to explicit many-body wavefunctions and construct their MPS representations step by step. We will consider a four-spin state, following the successive factorizations across the three bipartitions of the chain.

### 4.7.1 A Four-Spin Wavefunction

Consider the normalized state

$$
|\psi\rangle
=
\frac{1}{\sqrt{10}}
\left(
2|0000\rangle
+|0011\rangle
+|1100\rangle
+2|1111\rangle
\right).
$$

Our goal is to express this state as an MPS with four site tensors:

$$
\psi_{\sigma_1\sigma_2\sigma_3\sigma_4}
=
\sum_{\alpha_1,\alpha_2,\alpha_3}
A^{\sigma_1}_{\alpha_1}
A^{\sigma_2}_{\alpha_1\alpha_2}
A^{\sigma_3}_{\alpha_2\alpha_3}
A^{\sigma_4}_{\alpha_3}.
$$

We will construct the tensors through successive SVDs across the three cuts

$$
1|234,\qquad 12|34,\qquad 123|4.
$$

At each step, we factorize the coefficient tensor into a matrix, identify the next MPS tensor from its left singular vectors, and carry the remaining factors forward.

### 4.7.2 First Cut: $1|234$

We first reshape the coefficient tensor into a matrix whose row index is $\sigma_1$ and whose column index is $(\sigma_2,\sigma_3,\sigma_4)$.

Using the column ordering

$$
000,\ 001,\ 010,\ 011,\ 100,\ 101,\ 110,\ 111,
$$

we obtain

$$
C^{(1)}
=
\frac{1}{\sqrt{10}}
\begin{pmatrix}
2&0&0&1&0&0&0&0\\
0&0&0&0&1&0&0&2
\end{pmatrix}.
$$

The two rows are orthogonal and have equal squared norms. Consequently,

$$
C^{(1)}C^{(1)\dagger}
=
\begin{pmatrix}
\frac12&0\\
0&\frac12
\end{pmatrix}.
$$

Both singular values are therefore

$$
s_1^{(1)}=s_2^{(1)}=\frac{1}{\sqrt2}.
$$

We can choose the left singular vectors to be the standard basis vectors, so that

$$
U^{(1)}=I_2.
$$

The first MPS tensor is then

$$
A_{(1)}^0=(1,0),
\qquad
A_{(1)}^1=(0,1).
$$

The first bond dimension is

$$
\chi_1=2.
$$

Both singular values are nonzero, so retaining both is necessary for an exact representation across this cut.

### 4.7.3 Second Cut: $12|34$

We now reshape the remaining factors to separate the second spin from the rest. The row index is $(\alpha_1,\sigma_2)$, and the column index is $(\sigma_3,\sigma_4)$.

For this state, the resulting matrix is

$$
C^{(2)}
=
\frac{1}{\sqrt{10}}
\begin{pmatrix}
2&0&0&1\\
0&0&0&0\\
0&0&0&0\\
1&0&0&2
\end{pmatrix},
$$

where the row and column ordering is $00,01,10,11$.

The nonzero part of this matrix is the same matrix whose SVD we calculated in Section 4.4:

$$
C=
\frac{1}{\sqrt{10}}
\begin{pmatrix}
2&1\\
1&2
\end{pmatrix}.
$$

We can therefore use the singular values already obtained:

$$
s_1^{(2)}=\frac{3}{\sqrt{10}},
\qquad
s_2^{(2)}=\frac{1}{\sqrt{10}}.
$$

The corresponding left singular vectors, embedded in the four-dimensional basis, are

$$
|u_1\rangle
=
\frac{1}{\sqrt2}
\begin{pmatrix}
1\\0\\0\\1
\end{pmatrix},
\qquad
|u_2\rangle
=
\frac{1}{\sqrt2}
\begin{pmatrix}
1\\0\\0\\-1
\end{pmatrix}.
$$

These vectors determine the second MPS tensor when reshaped according to the indices $(\alpha_1,\sigma_2)$:

$$
A_{(2)}^0
=
\frac{1}{\sqrt2}
\begin{pmatrix}
1&1\\
0&0
\end{pmatrix},
$$

and

$$
A_{(2)}^1
=
\frac{1}{\sqrt2}
\begin{pmatrix}
0&0\\
1&-1
\end{pmatrix}.
$$

Each matrix has dimensions $2\times2$. Its row index is the incoming bond index $\alpha_1$, and its column index is the outgoing bond index $\alpha_2$.

Thus,

$$
\chi_2=2.
$$

An important detail is that the second tensor comes from the left singular vectors of the reshaped residual tensor. We cannot generally assign the singular vectors of a full bipartition directly to an individual site tensor; here, the identification works because the residual factorization has the same nonzero matrix structure.

### 4.7.4 Third Cut: $123|4$

We next separate the first three spins from the fourth. The row index is $(\sigma_1,\sigma_2,\sigma_3)$, and the column index is $\sigma_4$.

Using the row ordering

$$
000,\ 001,\ 010,\ 011,\ 100,\ 101,\ 110,\ 111,
$$

the coefficient matrix is

$$
C^{(3)}
=
\frac{1}{\sqrt{10}}
\begin{pmatrix}
2&0\\
0&1\\
0&0\\
0&0\\
0&0\\
0&0\\
1&0\\
0&2
\end{pmatrix}.
$$

Its columns are orthogonal and have equal squared norms:

$$
C^{(3)\dagger}C^{(3)}
=
\begin{pmatrix}
\frac12&0\\
0&\frac12
\end{pmatrix}.
$$

Therefore,

$$
s_1^{(3)}=s_2^{(3)}=\frac{1}{\sqrt2},
$$

and the final bond dimension is

$$
\chi_3=2.
$$

To obtain the third tensor, we reshape the residual matrix from the second SVD, grouping $(\alpha_2,\sigma_3)$ into the row index and $\sigma_4$ into the column index. The resulting matrix is

$$
M^{(3)}
=
\frac{1}{\sqrt{20}}
\begin{pmatrix}
3&0\\
0&3\\
1&0\\
0&-1
\end{pmatrix}.
$$

Its columns are orthogonal, with squared norms $1/2$. We can therefore choose the left singular vectors as the normalized columns of this matrix.

Reshaping those vectors gives

$$
A_{(3)}^0
=
\frac{1}{\sqrt{10}}
\begin{pmatrix}
3&0\\
1&0
\end{pmatrix},
$$

and

$$
A_{(3)}^1
=
\frac{1}{\sqrt{10}}
\begin{pmatrix}
0&3\\
0&-1
\end{pmatrix}.
$$

The final tensor contains the remaining right singular vectors and singular values. With the chosen SVD convention, it is

$$
A_{(4)}^0
=
\begin{pmatrix}
\frac{1}{\sqrt2}\\
0
\end{pmatrix},
\qquad
A_{(4)}^1
=
\begin{pmatrix}
0\\
\frac{1}{\sqrt2}
\end{pmatrix}.
$$

We have now obtained all four site tensors, with bond dimensions

$$
\chi_1=\chi_2=\chi_3=2.
$$

### 4.7.5 Reconstructing the Wavefunction

The complete MPS representation is

$$
\psi_{\sigma_1\sigma_2\sigma_3\sigma_4}
=
\sum_{\alpha_1,\alpha_2,\alpha_3}
A_{(1)}^{\sigma_1}
A_{(2)}^{\sigma_2}
A_{(3)}^{\sigma_3}
A_{(4)}^{\sigma_4},
$$

where the virtual indices are contracted between neighboring tensors.

Let us verify a few amplitudes explicitly. For $|0000\rangle$,

$$
\begin{aligned}
\psi_{0000}
&=
A_{(1)}^0
A_{(2)}^0
A_{(3)}^0
A_{(4)}^0\\
&=
\frac{2}{\sqrt{10}}.
\end{aligned}
$$

Similarly, for $|0011\rangle$,

$$
\begin{aligned}
\psi_{0011}
&=
A_{(1)}^0
A_{(2)}^0
A_{(3)}^1
A_{(4)}^1\\
&=
\frac{1}{\sqrt{10}}.
\end{aligned}
$$

The other nonzero amplitudes are

$$
\psi_{1100}=\frac{1}{\sqrt{10}},
\qquad
\psi_{1111}=\frac{2}{\sqrt{10}}.
$$

All other computational-basis amplitudes vanish. Contracting the MPS tensors therefore reproduces the original wavefunction exactly.

### 4.7.6 Summary

This example illustrates how successive SVDs turn a many-body coefficient tensor into a chain of smaller tensors. Each factorization introduces a virtual index, and the number of retained singular values determines the corresponding bond dimension.

For this four-spin state, all three cuts have rank two, so an exact MPS representation exists with bond dimension two throughout the chain.

In the implementation, we will construct the tensors programmatically and compare the reconstructed coefficient tensor with the original one. This will provide a direct numerical check of the successive-SVD procedure before we use MPS for ground-state calculations.

In [ ]:
# To be implemented

## 4.8 The Variational Principle for MPS

We have now learned how to represent a many-body wavefunction as a matrix product state. The next question is: **How can we use this representation to approximate the ground state of a quantum many-body Hamiltonian?**

For our spin-$\frac12$ Heisenberg chain, the Hamiltonian is

$$
H=\sum_{i=1}^{N-1}\mathbf S_i\cdot\mathbf S_{i+1},
$$

with $J=1$ and open boundary conditions.

Let $|\psi(A)\rangle$ denote an MPS whose tensors collectively are represented by $A$. Our goal is to find the tensors that give the lowest possible energy within the family of states we consider.

The energy expectation value of a normalized state is

$$
E=\langle\psi|H|\psi\rangle.
$$

However, during optimization, we need not impose normalization on every trial state explicitly. Instead, we can work with the energy functional

$$
E[A]
=
\frac{\langle\psi(A)|H|\psi(A)\rangle}
{\langle\psi(A)|\psi(A)\rangle}.
$$

This expression is the **Rayleigh quotient** of the Hamiltonian with respect to the trial state.

The variational principle guarantees that, for any nonzero trial state,

$$
E[A]\geq E_0,
$$

where $E_0$ is the exact ground-state energy of $H$. Equality holds when the trial state belongs to the ground-state eigenspace.

Our variational problem is therefore

$$
\boxed{
\min_A
\frac{\langle\psi(A)|H|\psi(A)\rangle}
{\langle\psi(A)|\psi(A)\rangle}
}
$$

subject to the chosen MPS structure and bond dimensions.

The bond dimensions determine the family of states accessible to the optimization. If they are sufficiently large, an exact representation may be possible; if they are restricted, the optimization searches for the lowest-energy state within the corresponding MPS family.

An important distinction is that the variational principle itself introduces no approximation: the energy bound is exact. The approximation enters when we restrict the trial states to an MPS family that cannot represent the exact ground state.

We now need to determine the mathematical condition that a state must satisfy to minimize this energy functional. To do so, we will examine the Rayleigh quotient and derive the corresponding eigenvalue equation using a Lagrange multiplier.

## 4.9 From the Rayleigh Quotient to the Eigenvalue Equation

Before considering the restrictions imposed by an MPS representation, let us determine what minimizing the energy means when the trial state can be any nonzero vector in the Hilbert space.

Write the state as

$$
|\psi\rangle=\sum_n c_n|n\rangle,
$$

where $\{|n\rangle\}$ is an orthonormal basis and the coefficients $c_n$ are collected into a vector $\mathbf c$. In this basis, the energy functional is

$$
E[\mathbf c]
=
\frac{\mathbf c^\dagger H\mathbf c}
{\mathbf c^\dagger\mathbf c}.
$$

We want to find the stationary points of this quotient. One way to handle the denominator is to impose normalization as a constraint. We therefore minimize $\langle\psi|H|\psi\rangle$ subject to $\langle\psi|\psi\rangle=1$.

Introducing a Lagrange multiplier $E$, define

$$
\mathcal L
=
\langle\psi|H|\psi\rangle
-
E\big(\langle\psi|\psi\rangle-1\big).
$$

In terms of the coefficient vector,

$$
\mathcal L
=
\mathbf c^\dagger H\mathbf c
-
E\big(\mathbf c^\dagger\mathbf c-1\big).
$$

Since the coefficients may be complex, we take the variation with respect to $\mathbf c^\dagger$, treating $\mathbf c$ and $\mathbf c^\dagger$ as independent variables for the purpose of differentiation. The stationarity condition is

$$
\frac{\partial\mathcal L}{\partial\mathbf c^\dagger}
=
H\mathbf c-E\mathbf c
=
0.
$$

Consequently,

$$
\boxed{H\mathbf c=E\mathbf c.}
$$

The stationary states of the energy functional are therefore eigenstates of the Hamiltonian, and their stationary energies are the corresponding eigenvalues. Minimizing the Rayleigh quotient selects the smallest eigenvalue, $E_0$, with a corresponding ground-state eigenvector.

This result gives us an important reference point. If we optimize over the entire Hilbert space, the variational problem recovers the exact ground state. In an MPS calculation, however, the state is constrained to have a particular tensor-network structure and specified bond dimensions. The central question is now: **what changes when the variational parameters are the entries of MPS tensors rather than the unrestricted coefficients of the wavefunction?**

## 4.10 Variational Optimization within the MPS Family

For a general state, the variational parameters are the coefficients $c_n$ of its expansion in a Hilbert-space basis. The stationarity condition then gives the full eigenvalue equation,

$$
H|\psi\rangle=E|\psi\rangle.
$$

An MPS introduces a more structured parameterization. Instead of optimizing all $2^N$ amplitudes independently, we represent them through local tensors:

$$
|\psi(A)\rangle
=
\sum_{\sigma_1,\ldots,\sigma_N}
A_{(1)}^{\sigma_1}A_{(2)}^{\sigma_2}\cdots A_{(N)}^{\sigma_N}
|\sigma_1\sigma_2\cdots\sigma_N\rangle.
$$

The entries of the tensors $A_{(i)}^{\sigma_i}$ are now the variational parameters. The energy to minimize remains the Rayleigh quotient,

$$
E[A]
=
\frac{\langle\psi(A)|H|\psi(A)\rangle}
{\langle\psi(A)|\psi(A)\rangle}.
$$

The crucial difference is that the amplitudes of $|\psi(A)\rangle$ are no longer independent: they are linked by the MPS factorization. Consequently, varying one tensor generally changes many wavefunction amplitudes simultaneously.

The optimization problem is therefore

$$
\boxed{
\min_{\{A_{(i)}^{\sigma_i}\}}
E[A]
}
$$

with the prescribed tensor dimensions and boundary structure held fixed.

Unlike the unrestricted problem, stationarity with respect to the tensor entries does not, in general, imply the full equation $H|\psi\rangle=E|\psi\rangle$. It implies stationarity only with respect to variations accessible through the chosen MPS parameterization. The exact ground state is recovered only if the allowed MPS family contains a ground-state vector and the global minimum is found.

This raises a practical question: **how can we optimize a collection of tensors efficiently when the energy depends on all of them at once?** Rather than changing every tensor simultaneously, we can hold most tensors fixed and optimize one tensor at a time. This is the basic idea behind the variational optimization used in DMRG.

## 4.11 Local Variational Optimization: From a Three-Spin Chain to a Generalized Eigenvalue Problem

Consider a three-spin MPS,

$$
|\psi\rangle
=
\sum_{\sigma_1,\sigma_2,\sigma_3}
A^{\sigma_1}B^{\sigma_2}C^{\sigma_3}
|\sigma_1\sigma_2\sigma_3\rangle.
$$

Our aim is to minimize the energy by varying $B$ while keeping $A$ and $C$ fixed.

Why does this suggest introducing a new vector of variational parameters?

For each physical configuration $(\sigma_1,\sigma_2,\sigma_3)$, the wavefunction amplitude is

$$
\psi_{\sigma_1\sigma_2\sigma_3}
=
A^{\sigma_1}B^{\sigma_2}C^{\sigma_3}.
$$

Once $A$ and $C$ are fixed, this amplitude is a linear function of the entries of $B$. The same is true of the entire wavefunction: every coefficient in the computational basis depends linearly on those entries.

To make this dependence explicit, suppose the bond dimensions on either side of $B$ are $D_L$ and $D_R$. The middle tensor has entries

$$
B^{\sigma_2}_{\alpha\beta},
\qquad
\sigma_2\in\{0,1\},\quad
\alpha=1,\ldots,D_L,\quad
\beta=1,\ldots,D_R.
$$

There are $2D_LD_R$ such entries. Collect them into a single vector $\mathbf b$, with components

$$
b_{(\sigma_2,\alpha,\beta)}
=
B^{\sigma_2}_{\alpha\beta}.
$$

The particular ordering of these components is arbitrary, provided we use it consistently.

### Constructing the states $|\Phi_{\sigma_2,\alpha,\beta}\rangle$

We can now ask: what wavefunction results from changing just one entry $B^{\sigma_2}_{\alpha\beta}$ while keeping every other entry of $B$ equal to zero?

The resulting state is precisely the state associated with that entry. More explicitly, define

$$
|\Phi_{\sigma_2,\alpha,\beta}\rangle
=
\sum_{\sigma_1,\sigma_3}
A^{\sigma_1}_{\alpha}
C^{\sigma_3}_{\beta}
|\sigma_1\sigma_2\sigma_3\rangle.
$$

Here, $A^{\sigma_1}_{\alpha}$ denotes the $\alpha$th component of the boundary vector $A^{\sigma_1}$, and $C^{\sigma_3}_{\beta}$ denotes the $\beta$th component of the boundary vector $C^{\sigma_3}$.

Notice what this definition means. The state $|\Phi_{\sigma_2,\alpha,\beta}\rangle$ contains the contributions from all physical configurations compatible with the selected physical index $\sigma_2$ and the selected bond indices $\alpha,\beta$. Its amplitudes are determined entirely by the fixed tensors $A$ and $C$.

The original wavefunction can therefore be reconstructed by summing these states, weighted by the corresponding entries of $B$:

$$
|\psi\rangle
=
\sum_{\sigma_2,\alpha,\beta}
B^{\sigma_2}_{\alpha\beta}
|\Phi_{\sigma_2,\alpha,\beta}\rangle.
$$

Equivalently, using the combined index $\mu=(\sigma_2,\alpha,\beta)$, we obtain

$$
\boxed{
|\psi(\mathbf b)\rangle
=
\sum_\mu b_\mu|\Phi_\mu\rangle.
}
$$

This is why we introduced the vector $\mathbf b$: it allows us to express the MPS as a linear combination of fixed states, with the entries of the tensor being the expansion coefficients.

The important difference from an ordinary expansion in the computational basis is that the states $|\Phi_\mu\rangle$ are not the computational basis states themselves. They are constructed from the fixed tensors surrounding the tensor being optimized. Consequently, they need not be orthogonal or normalized.

This is the origin of the Gram matrix that will appear in the local variational problem.

### Constructing the Effective Hamiltonian and Gram Matrix

We have expressed the three-spin state as

$$
|\psi(\mathbf b)\rangle
=
\sum_\mu b_\mu|\Phi_\mu\rangle,
$$

where the states $|\Phi_\mu\rangle$ are fixed because the tensors $A$ and $C$ are held constant.

Let us first evaluate the energy numerator. Substituting the expansion into the expectation value gives

$$
\begin{aligned}
\langle\psi|H|\psi\rangle
&=
\left(\sum_\mu b_\mu^*\langle\Phi_\mu|\right)
H
\left(\sum_\nu b_\nu|\Phi_\nu\rangle\right)\\
&=
\sum_{\mu,\nu}
b_\mu^*b_\nu
\langle\Phi_\mu|H|\Phi_\nu\rangle.
\end{aligned}
$$

This expression is quadratic in the entries of $B$. To write it compactly, define the effective Hamiltonian by its matrix elements,

$$
(H_{\mathrm{eff}})_{\mu\nu}
=
\langle\Phi_\mu|H|\Phi_\nu\rangle.
$$

The energy numerator then becomes

$$
\boxed{
\langle\psi|H|\psi\rangle
=
\mathbf b^\dagger H_{\mathrm{eff}}\mathbf b.
}
$$

The word *effective* indicates that this matrix describes the action of the physical Hamiltonian within the variational states generated by the fixed surrounding tensors. It is not, in general, the original Hamiltonian matrix in the computational basis.

Now consider the denominator of the Rayleigh quotient. The squared norm is

$$
\begin{aligned}
\langle\psi|\psi\rangle
&=
\left(\sum_\mu b_\mu^*\langle\Phi_\mu|\right)
\left(\sum_\nu b_\nu|\Phi_\nu\rangle\right)\\
&=
\sum_{\mu,\nu}
b_\mu^*b_\nu
\langle\Phi_\mu|\Phi_\nu\rangle.
\end{aligned}
$$

This motivates the definition of the Gram matrix,

$$
G_{\mu\nu}
=
\langle\Phi_\mu|\Phi_\nu\rangle.
$$

Consequently,

$$
\boxed{
\langle\psi|\psi\rangle
=
\mathbf b^\dagger G\mathbf b.
}
$$

The Gram matrix records the inner products among the variational states. If those states are orthonormal, $G$ is the identity; otherwise, it accounts for their nonorthogonality.

Combining the numerator and denominator, the energy as a function of the middle tensor is

$$
\boxed{
E(\mathbf b)
=
\frac{\mathbf b^\dagger H_{\mathrm{eff}}\mathbf b}
{\mathbf b^\dagger G\mathbf b}.
}
$$

### Deriving the Generalized Eigenvalue Problem

We now minimize this quotient. Equivalently, we minimize the energy numerator while constraining the norm to be one:

$$
\mathbf b^\dagger G\mathbf b=1.
$$

Introduce a Lagrange multiplier $E$ and define

$$
\mathcal L(\mathbf b)
=
\mathbf b^\dagger H_{\mathrm{eff}}\mathbf b
-
E\left(\mathbf b^\dagger G\mathbf b-1\right).
$$

Taking the variation with respect to $\mathbf b^\dagger$ yields

$$
\frac{\partial\mathcal L}{\partial\mathbf b^\dagger}
=
H_{\mathrm{eff}}\mathbf b-EG\mathbf b
=
0.
$$

We therefore obtain

$$
\boxed{
H_{\mathrm{eff}}\mathbf b
=
E G\mathbf b.
}
$$

The local variational problem is a generalized eigenvalue problem because the variational states $|\Phi_\mu\rangle$ are not necessarily orthonormal.

This brings us to a natural question: **can we exploit the freedom in our MPS representation to make the variational states orthonormal, so that $G=I$ and the generalized eigenvalue problem becomes an ordinary one?**

## 4.12 Gauge Freedom in Matrix Product States

We want to find an MPS representation in which the variational states $|\Phi_\mu\rangle$ are orthonormal. Before investigating how to achieve this, we need to understand an important property of MPS representations: **different sets of tensors can represent exactly the same physical wavefunction.**

### Inserting an identity between neighboring tensors

Consider two neighboring tensors $A^{\sigma_i}$ and $A^{\sigma_{i+1}}$. Their contribution to a wavefunction amplitude contains the contraction

$$
A^{\sigma_i}A^{\sigma_{i+1}}.
$$

Let $X$ be any invertible matrix acting on their shared bond. Since

$$
XX^{-1}=I,
$$

we may write

$$
A^{\sigma_i}A^{\sigma_{i+1}}
=
A^{\sigma_i}XX^{-1}A^{\sigma_{i+1}}.
$$

Define new tensors

$$
\widetilde A^{\sigma_i}=A^{\sigma_i}X,
\qquad
\widetilde A^{\sigma_{i+1}}=X^{-1}A^{\sigma_{i+1}}.
$$

Their contraction is unchanged:

$$
\widetilde A^{\sigma_i}\widetilde A^{\sigma_{i+1}}
=
A^{\sigma_i}XX^{-1}A^{\sigma_{i+1}}
=
A^{\sigma_i}A^{\sigma_{i+1}}.
$$

Thus, inserting $X$ and $X^{-1}$ on an internal bond and absorbing them into the neighboring tensors leaves the physical wavefunction invariant.

The same transformation can be performed on any internal bond of an MPS. Although the individual tensors change, every physical amplitude remains the same. This freedom is called **MPS gauge freedom**.

### Applying gauge freedom to the three-spin chain

Return to our three-spin state,

$$
|\psi\rangle
=
\sum_{\sigma_1,\sigma_2,\sigma_3}
A^{\sigma_1}B^{\sigma_2}C^{\sigma_3}
|\sigma_1\sigma_2\sigma_3\rangle.
$$

We can insert an invertible matrix $X$ on the bond between $A$ and $B$:

$$
A^{\sigma_1}B^{\sigma_2}C^{\sigma_3}
=
(A^{\sigma_1}X)(X^{-1}B^{\sigma_2})C^{\sigma_3}.
$$

Likewise, inserting an invertible matrix $Y$ on the bond between $B$ and $C$ gives

$$
A^{\sigma_1}B^{\sigma_2}C^{\sigma_3}
=
A^{\sigma_1}(B^{\sigma_2}Y)(Y^{-1}C^{\sigma_3}).
$$

We can therefore redistribute invertible transformations across the internal bonds without changing the represented state.

This is useful because the variational states $|\Phi_{\sigma_2,\alpha,\beta}\rangle$ depend on the tensors $A$ and $C$ held fixed during the optimization of $B$. Changing the MPS gauge changes these surrounding tensors and hence changes the variational basis, even though the complete physical state represented by the full MPS remains unchanged.

We can now pose the question that motivates canonicalization:

**Can we use this gauge freedom to choose the surrounding tensors so that the variational states $|\Phi_{\sigma_2,\alpha,\beta}\rangle$ become orthonormal?**

If we can, their Gram matrix will satisfy

$$
G_{\mu\nu}=\delta_{\mu\nu},
$$

and therefore

$$
G=I.
$$

The local generalized eigenvalue problem would then reduce to the ordinary eigenvalue problem,

$$
H_{\mathrm{eff}}\mathbf b=E\mathbf b.
$$

To determine the conditions required for this orthonormality, we will return to the explicit construction of the states $|\Phi_{\sigma_2,\alpha,\beta}\rangle$ and examine their inner products.

## 4.13 Deriving the Orthonormality Conditions

Recall the variational states constructed by varying the middle tensor of a three-spin MPS:

$$
|\Phi_{\sigma_2,\alpha,\beta}\rangle
=
\sum_{\sigma_1,\sigma_3}
A^{\sigma_1}_{\alpha}
C^{\sigma_3}_{\beta}
|\sigma_1\sigma_2\sigma_3\rangle.
$$

Here, $\sigma_2$ labels the physical state of the middle spin, while $\alpha$ and $\beta$ label the bond indices connecting the middle tensor to its left and right neighbors.

Our goal is to determine when these states form an orthonormal set. Consider two such states, labelled by $(\sigma_2,\alpha,\beta)$ and $(\tau_2,\gamma,\delta)$. Their inner product is

$$
\begin{aligned}
&\langle\Phi_{\sigma_2,\alpha,\beta}
\mid
\Phi_{\tau_2,\gamma,\delta}\rangle\\
&=
\sum_{\sigma_1,\sigma_3}
\sum_{\upsilon_1,\upsilon_3}
\left(A^{\upsilon_1}_{\gamma}\right)^*
\left(C^{\upsilon_3}_{\delta}\right)^*
A^{\sigma_1}_{\alpha}
C^{\sigma_3}_{\beta}
\langle \upsilon_1\tau_2\upsilon_3\mid
\sigma_1\sigma_2\sigma_3\rangle.
\end{aligned}
$$

The computational basis states are orthonormal, so

$$
\langle \upsilon_1\tau_2\upsilon_3\mid
\sigma_1\sigma_2\sigma_3\rangle
=
\delta_{\upsilon_1\sigma_1}
\delta_{\tau_2\sigma_2}
\delta_{\upsilon_3\sigma_3}.
$$

The inner product therefore simplifies to

$$
\begin{aligned}
&\langle\Phi_{\sigma_2,\alpha,\beta}
\mid
\Phi_{\tau_2,\gamma,\delta}\rangle\\
&=
\delta_{\sigma_2\tau_2}
\left(\sum_{\sigma_1}
(A^{\sigma_1}_{\gamma})^*A^{\sigma_1}_{\alpha}\right)
\left(\sum_{\sigma_3}
(C^{\sigma_3}_{\delta})^*C^{\sigma_3}_{\beta}\right).
\end{aligned}
$$

This factorization is the key result. The middle physical index already contributes the factor $\delta_{\sigma_2\tau_2}$. The remaining inner product separates into a factor determined by the left tensor and a factor determined by the right tensor.

### The condition on the left tensor

For the left factor to give $\delta_{\alpha\gamma}$, we require

$$
\sum_{\sigma_1}
(A^{\sigma_1}_{\gamma})^*A^{\sigma_1}_{\alpha}
=
\delta_{\gamma\alpha}.
$$

Equivalently,

$$
\boxed{
\sum_{\sigma_1}(A^{\sigma_1})^\dagger A^{\sigma_1}=I.
}
$$

This is the left-orthonormality condition for the first tensor, when $A^{\sigma_1}$ is viewed as a row vector. It says that the rows formed by combining the physical and bond indices are orthonormal.

### The condition on the right tensor

Similarly, for the right factor to give $\delta_{\beta\delta}$, we require

$$
\sum_{\sigma_3}
(C^{\sigma_3}_{\delta})^*C^{\sigma_3}_{\beta}
=
\delta_{\delta\beta}.
$$

Equivalently,

$$
\boxed{
\sum_{\sigma_3}C^{\sigma_3}(C^{\sigma_3})^\dagger=I.
}
$$

This is the right-orthonormality condition for the last tensor, when $C^{\sigma_3}$ is viewed as a column vector. It says that the columns formed by combining the physical and bond indices are orthonormal.

When both conditions hold, the inner product becomes

$$
\langle\Phi_{\sigma_2,\alpha,\beta}
\mid
\Phi_{\tau_2,\gamma,\delta}\rangle
=
\delta_{\sigma_2\tau_2}
\delta_{\alpha\gamma}
\delta_{\beta\delta}.
$$

Thus,

$$
\boxed{G=I.}
$$

The generalized eigenvalue problem for the middle tensor now reduces to

$$
\boxed{
H_{\mathrm{eff}}\mathbf b=E\mathbf b.
}
$$

We have therefore derived the desired result: if the tensors surrounding the optimized tensor satisfy the appropriate orthonormality conditions, the induced variational states are orthonormal, and the local norm matrix becomes the identity.

The remaining question is how to impose these conditions on a general MPS. In particular, can we transform an arbitrary MPS into a representation in which all tensors to the left of the optimized site are left-orthonormal and all tensors to its right are right-orthonormal?

## 4.14 Left- and Right-Orthonormal MPS Tensors

In the three-spin example, we found that the variational states become orthonormal when the tensors surrounding the tensor being optimized satisfy suitable orthonormality conditions. We now generalize these conditions to an arbitrary MPS.

Consider a tensor $A_{(i)}$ at site $i$. It connects the bond to its left, the physical state of the spin at site $i$, and the bond to its right.

The key idea is to regard the tensor as a matrix in two different ways, depending on which indices we group together.

### Left-orthonormality

First, combine the left bond index and the physical index into a single index. The tensor can then be viewed as a matrix whose columns are labelled by the right bond index.

We call the tensor left-orthonormal when these columns are orthonormal. In matrix notation, this means

$$
M^\dagger M=I.
$$

Written in terms of the original MPS tensor, the condition becomes

$$
\boxed{
\sum_{\sigma_i}
\left(A_{(i)}^{\sigma_i}\right)^\dagger
A_{(i)}^{\sigma_i}
=
I.
}
$$

This condition ensures that contracting the physical and left-bond indices leaves an identity on the right bond.

### Right-orthonormality

Alternatively, combine the physical index and the right bond index into a single index. The tensor is now viewed as a matrix whose rows are labelled by the left bond index.

We call the tensor right-orthonormal when these rows are orthonormal. The corresponding matrix condition is

$$
MM^\dagger=I,
$$

or, in terms of the MPS tensor,

$$
\boxed{
\sum_{\sigma_i}
A_{(i)}^{\sigma_i}
\left(A_{(i)}^{\sigma_i}\right)^\dagger
=
I.}
$$

This condition ensures that contracting the physical and right-bond indices leaves an identity on the left bond.

### Why do we need both conditions?

Left- and right-orthonormality simplify contractions in opposite directions. This is exactly what we need when optimizing a tensor while keeping the rest of the MPS fixed.

If the tensors to the left of the optimized site are left-orthonormal and those to its right are right-orthonormal, their contributions to the inner products of the local variational states reduce to identity matrices. Consequently, those variational states are orthonormal, and the local Gram matrix becomes

$$
G=I.
$$

The next question is how to transform an arbitrary MPS into a representation satisfying these conditions without changing its physical wavefunction.

## 4.15 Using QR Decomposition to Orthonormalize an MPS

We now know which orthonormality conditions we want our MPS tensors to satisfy. The question is how to obtain them without changing the physical state.

The key is to reshape a tensor into a matrix and apply QR decomposition.

### From a tensor to an orthonormal matrix

Consider a tensor $A_{(i)}$ at site $i$. By combining its physical index with its left bond index, we can reshape it into a matrix $M$. A QR decomposition gives

$$
M=QR,
$$

where the columns of $Q$ are orthonormal,

$$
Q^\dagger Q=I.
$$

Thus, $Q$ provides precisely the orthonormal structure we want. We can reshape it back into a tensor and use it in place of $A_{(i)}$.

But we cannot simply discard $R$, because that would generally change the wavefunction. Instead, we absorb $R$ into the tensor at the next site. The contraction of the two tensors remains unchanged:

$$
M A_{(i+1)}
=
Q\big(R A_{(i+1)}\big).
$$

The physical state is therefore preserved, even though the individual tensors have changed.

### Repeating the procedure

We can apply this procedure successively from left to right, making each processed tensor left-orthonormal and transferring the remaining factor to its neighbor.

A corresponding procedure, applied from right to left, produces right-orthonormal tensors.

This illustrates how MPS gauge freedom becomes practically useful: **we can redistribute factors between neighboring tensors to impose orthonormality without changing the represented physical state.**

We can now use these two procedures together to prepare an MPS for local variational optimization.

## 4.16 Mixed Canonical Form and Local Optimization

Suppose we want to optimize the tensor at site $i$. We can use the procedures from the previous section to arrange the MPS so that all tensors to its left are left-orthonormal, while all tensors to its right are right-orthonormal.

Schematically, the state has the form

$$
|\psi\rangle
=
\sum_{\sigma_1,\ldots,\sigma_N}
A_{(1)}^{\sigma_1}\cdots
A_{(i-1)}^{\sigma_{i-1}}
B_{(i)}^{\sigma_i}
C_{(i+1)}^{\sigma_{i+1}}\cdots
C_{(N)}^{\sigma_N}
|\sigma_1\cdots\sigma_N\rangle.
$$

The tensors $A_{(j)}$ on the left satisfy the left-orthonormality condition, while the tensors $C_{(j)}$ on the right satisfy the right-orthonormality condition. The central tensor $B_{(i)}$ is the tensor we optimize.

This arrangement is called the **mixed canonical form**.

### Why is this useful?

Recall why we introduced the variational states $|\Phi_\mu\rangle$: changing the central tensor produces a linear combination of these states, and their inner products determine the Gram matrix $G$ in the local energy functional,

$$
E(\mathbf b)
=
\frac{\mathbf b^\dagger H_{\mathrm{eff}}\mathbf b}
{\mathbf b^\dagger G\mathbf b}.
$$

When the surrounding tensors satisfy the appropriate orthonormality conditions, their contributions to these inner products reduce to identity matrices. The variational states are then orthonormal, so

$$
G=I.
$$

The local optimization problem consequently simplifies to

$$
\boxed{
H_{\mathrm{eff}}\mathbf b=E\mathbf b.
}
$$

Instead of solving a generalized eigenvalue problem, we can solve an ordinary eigenvalue problem for the effective Hamiltonian.

### Moving the optimization site

After optimizing the central tensor, we can move the optimization site to a neighboring position. A QR decomposition transfers the remaining factor into the neighboring tensor while preserving the physical state. The orthonormal structure is updated accordingly, allowing the next tensor to be optimized.

Repeating this process from one end of the chain to the other, and then back again, produces a sequence of local variational updates. This is the basic idea behind a one-site optimization sweep.

We have now seen why canonical forms matter: they are not merely alternative ways to write an MPS. They provide an orthonormal local variational basis, simplifying the optimization problem.

## 4.17 The DMRG Sweep Algorithm

We have established that, in mixed canonical form, optimizing a single MPS tensor reduces to solving an ordinary eigenvalue problem for an effective Hamiltonian. We can now turn this result into an iterative algorithm.

The central idea of DMRG is to optimize one tensor at a time while maintaining the canonical structure of the MPS. After optimizing a tensor, we move the optimization site to a neighboring position, update the canonical form, and repeat the procedure.

### A single local update

Suppose the current optimization site is $i$, with the MPS in mixed canonical form around that site. A local update consists of the following steps:

1. **Construct the effective Hamiltonian.** Contract the Hamiltonian with the fixed tensors to the left and right of site $i$. This produces an operator acting on the variational degrees of freedom of the central tensor.

2. **Solve the local eigenvalue problem.** Find the lowest-energy eigenvector of the effective Hamiltonian. Reshape this vector into the updated tensor $B_{(i)}^{\sigma_i}$.

3. **Move the optimization site.** Factorize the updated tensor using QR decomposition and absorb the remaining factor into the neighboring tensor. This transfers the optimization site while preserving the represented wavefunction.

4. **Restore the canonical structure.** The tensors already passed during the sweep retain their appropriate orthonormality, while the neighboring tensor becomes the new optimization site.

These steps produce a new locally optimized MPS representation. The process can then continue at the next site.

### Sweeping through the chain

A complete sweep moves the optimization site across the chain, typically from left to right. Once the right boundary is reached, the direction is reversed, and the optimization site travels back toward the left boundary.

During these sweeps, each local update improves the energy or leaves it unchanged, provided the local problem is solved for its lowest eigenvalue and the update is performed consistently. The energy is bounded below by the ground-state energy, so repeated updates can approach a variational minimum.

The sequence of updates is therefore

$$
\text{left-to-right sweep}
\quad\longrightarrow\quad
\text{right-to-left sweep}
\quad\longrightarrow\quad
\cdots
$$

The procedure is repeated until a chosen convergence criterion is satisfied, such as a sufficiently small change in energy between successive sweeps.

### What determines the quality of the result?

Two distinct issues affect the outcome.

First, the bond dimension $D$ determines the expressive power of the MPS. A larger bond dimension allows the state to represent more entanglement, but increases the computational cost.

Second, the optimization itself may converge to a local variational minimum rather than the global ground state. The result can depend on the initial MPS and the optimization strategy.

Thus, convergence of the energy alone does not prove that the exact ground state has been obtained. We must also assess the effect of the bond dimension and, where appropriate, compare against independent results.

We now have the conceptual structure of a DMRG sweep. The next step is to determine how the effective Hamiltonian can be constructed and applied efficiently without repeatedly forming the full many-body Hamiltonian matrix.